# Анализ электропотребления процессов рудоподготовки

**Назначение.** Полный пайплайн обработки многолетних данных по электропотреблению вводов, собственной генерации и инструментальных измерений на 6 кВ для подготовки числовых результатов и иллюстраций к статье в *Scientific Reports*.

**Содержимое:**

1. Установка зависимостей.
2. Конфигурация (одна ячейка — задаёте путь к папке с файлами).
3. Автоматическое обнаружение и классификация входных файлов по именам.
4. Универсальный загрузчик `xls/xlsx/csv` — обрабатывает разные структуры (помесячные таблицы, почасовые с интервальными метками, многолистовые экспорты Энерготестера / ПКК-57).
5. Описательная статистика, тесты согласия (Шапиро–Уилк, Колмогоров–Смирнов, Андерсон–Дарлинг), тесты стационарности (ADF, KPSS).
6. Визуализации: временные ряды, гистограммы, boxplot по сезонам / часам, STL-декомпозиция, ACF / PACF.
7. Кросс-секционный анализ: панели по предприятию, корреляционные матрицы, удельные веса.
8. Анализ инструментальных измерений на мельницах (синхронизация P, Q, расхода руды).
9. Расчёт удельного расхода электроэнергии (СУЭ).
10. Feature engineering для прогнозных моделей.
11. Бенчмарк 20+ моделей машинного обучения (TimeSeriesSplit, MAE / RMSE / MAPE / R²).
12. SHAP-анализ интерпретируемости лучшей модели.
13. Подбор гиперпараметров через Optuna.
14. Анализ асимметрии «генерация — нагрузка» (ключевой результат для статьи).
15. Количественная оценка резервов энергосбережения.
16. Экспорт всех результатов (CSV + PNG) в одну папку.

**Запуск:** ячейки выполняются последовательно. Если какой-то источник данных отсутствует, соответствующий блок пропускается с информационным сообщением.

**Окружение:** Google Colab или локальный Jupyter; код одинаков.


## 1. Установка пакетов

В Google Colab большинство пакетов уже предустановлены. Доустанавливаем то, чего может не быть.

In [ ]:
import sys, subprocess

def _pip(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Базовые
_pip(["pandas>=2.0", "numpy", "matplotlib", "seaborn", "scipy",
      "scikit-learn>=1.3", "statsmodels", "openpyxl", "xlrd>=2.0.1",
      "lxml", "html5lib", "beautifulsoup4"])  # последние три - для pd.read_html()

# Ускорители ML
_pip(["xgboost", "lightgbm", "catboost"])

# Интерпретация и тюнинг
_pip(["shap", "optuna"])

print("Все пакеты установлены.")


## 2. Конфигурация

Это **единственная ячейка**, которую обычно нужно отредактировать перед запуском: укажите путь к папке с входными xlsx / xls / csv-файлами и путь для сохранения результатов.

**Как загрузить данные в Colab:**

1. На левой панели Colab откройте вкладку «Файлы» (значок папки).
2. Создайте папку `input` внутри `/content/sample_data/` (или просто загрузите файлы прямо в `/content/sample_data/input/` через drag-and-drop).
3. Все исходные xlsx / xls / csv-файлы поместите в эту папку — имена файлов могут быть произвольными.

**Пример структуры:**

```
/content/sample_data/
├── input/             # все исходные файлы (имя не важно)
│   ├── ПС Ц-Р Ввод 1 по июль 2023.xlsx
│   ├── ПС Ц-Р яч.307 Фабрика по декабрь 2023.xlsx
│   ├── 16.07.2024 06.28.31-…xlsx
│   ├── ruda_1.xlsx
│   └── графики помесячно.xlsx
└── output/            # сюда сохранятся CSV + PNG (создастся автоматически)
```

Скрипт сам разберёт, какой файл к какому источнику относится, по ключевым словам в имени.

In [ ]:
from pathlib import Path

# ============ ОТРЕДАКТИРУЙТЕ ПОД СВОЙ ПУТЬ ============
INPUT_DIR  = Path("/content/sample_data/input")
OUTPUT_DIR = Path("/content/sample_data/output")
# =====================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "figures").mkdir(exist_ok=True)
(OUTPUT_DIR / "tables").mkdir(exist_ok=True)
(OUTPUT_DIR / "ml").mkdir(exist_ok=True)

print(f"INPUT_DIR  = {INPUT_DIR}")
print(f"OUTPUT_DIR = {OUTPUT_DIR}")
print(f"INPUT_DIR существует: {INPUT_DIR.exists()}")

if INPUT_DIR.exists():
    files = sorted(p for p in INPUT_DIR.iterdir() if p.is_file())
    print(f"Файлов в INPUT_DIR: {len(files)}\n")
    for p in files[:50]:
        print(f"   {p.name}  ({p.stat().st_size/1024:.0f} KB)")
else:
    print("\n!!! INPUT_DIR не найден. Создайте папку и загрузите туда файлы:")
    print(f"   - на левой панели Colab откройте вкладку «Файлы»")
    print(f"   - перейдите в /content/sample_data/")
    print(f"   - создайте папку 'input' и загрузите туда xlsx/xls/csv-файлы")
    print(f"   - либо измените INPUT_DIR выше на свой путь")


## 4. Глобальные импорты и стиль графиков

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")
sns.set_palette("deep")
plt.rcParams.update({
    "figure.dpi":       100,
    "savefig.dpi":      300,
    "savefig.bbox":     "tight",
    "font.size":        11,
    "axes.titlesize":   12,
    "axes.labelsize":   11,
    "legend.fontsize":  10,
    "xtick.labelsize":  10,
    "ytick.labelsize":  10,
})

SEED = 42
np.random.seed(SEED)
print("Импорты и стиль настроены.")


## 5. Автоматическое обнаружение и классификация файлов

Каждому файлу в `INPUT_DIR` присваивается:

| Поле | Описание |
|---|---|
| `kind` | один из `"bus_monthly"`, `"bus_hourly"`, `"factory_cell_monthly"`, `"factory_cell_hourly"`, `"generation_hourly"`, `"mill_measurement"`, `"ore_throughput"`, `"unknown"` |
| `entity` | какой объект описан: `bus_1`, `bus_2`, `cell_307`, `cell_402`, `gen_cell_2`, `gen_cell_23`, `mill_xxxx`, `ore` |
| `period` | `july` / `december` / `monthly` / дата сеанса измерения |
| `granularity` | `"monthly"`, `"hourly"`, `"sub_hourly"` |

Классификация — по регулярным выражениям к **нижнему регистру** имени файла, чтобы быть устойчивой к разным версиям имён.

In [ ]:
def classify_filename(name: str) -> dict:
    """Определяет, какому источнику данных соответствует файл по его имени."""
    n = name.lower().replace(" ", "")
    info = {"name": name, "kind": "unknown", "entity": None,
            "period": None, "granularity": None}

    # 1) данные по руде
    if "ruda" in n or "руда" in n:
        info.update(kind="ore_throughput", entity="ore", granularity="hourly")
        return info

    # 2) Файлы с датой ДД.ММ.ГГГГ_ЧЧ.ММ.СС в имени → инструментальные измерения мельницы
    if re.search(r"\d{2}\.\d{2}\.\d{4}\s*\d{2}\.\d{2}", name):
        info.update(kind="mill_measurement", entity="mill_session",
                    granularity="sub_hourly")
        # извлечь сессию (стартовую отметку) - удобно для ключа
        m = re.search(r"(\d{2}\.\d{2}\.\d{4})\s*(\d{2}\.\d{2}\.\d{2})", name)
        if m:
            info["period"] = f"{m.group(1)} {m.group(2)}"
        return info
    if re.match(r"^\d{6}_\d{6}", name):  # 160724_170724.xls
        info.update(kind="mill_measurement", entity="mill_session",
                    granularity="sub_hourly")
        info["period"] = name[:13]
        return info
    if "результаты_измерений" in n or "результатыизмерений" in n:
        info.update(kind="mill_measurement", entity="mill_summary",
                    granularity="sub_hourly")
        return info

    # 3) Сводные графики
    if "графики" in n and "помесячно" in n:
        info.update(kind="bus_monthly_combined", entity="all", granularity="monthly")
        return info

    # 4) ПС Ц-Р: вводы и ячейки
    if "ввод1" in n or ("ввод" in n and "1" in n.split("ввод")[1][:3]):
        ent = "bus_1"
    elif "ввод2" in n or ("ввод" in n and "2" in n.split("ввод")[1][:3]):
        ent = "bus_2"
    elif "яч.307" in n or "яч307" in n or "яч.3.07" in n:
        ent = "cell_307"
    elif "яч.402" in n or "яч402" in n:
        ent = "cell_402"
    elif "яч.2" in n or "яч2" in n:
        # отделим яч.2 от яч.23
        if "яч.23" in n or "яч23" in n:
            ent = "gen_cell_23"
        else:
            ent = "gen_cell_2"
    elif "яч.23" in n or "яч23" in n:
        ent = "gen_cell_23"
    else:
        ent = None

    # период
    if "июл" in n or "july" in n:
        period = "july"
    elif "дек" in n or "december" in n:
        period = "december"
    elif "помес" in n or "месяч" in n or "monthly" in n:
        period = "monthly"
    else:
        period = None

    # гранулярность
    granularity = "monthly" if period == "monthly" else "hourly"

    # тип
    if ent in ("bus_1", "bus_2"):
        kind = "bus_monthly" if granularity == "monthly" else "bus_hourly"
    elif ent in ("cell_307", "cell_402"):
        kind = "factory_cell_monthly" if granularity == "monthly" else "factory_cell_hourly"
    elif ent in ("gen_cell_2", "gen_cell_23"):
        kind = "generation_monthly" if granularity == "monthly" else "generation_hourly"
    else:
        kind = "unknown"

    info.update(kind=kind, entity=ent, period=period, granularity=granularity)
    return info


def discover_files(input_dir: Path) -> pd.DataFrame:
    """Сканирует директорию и возвращает DataFrame с классификацией."""
    rows = []
    for p in sorted(input_dir.iterdir()):
        if not p.is_file():
            continue
        if p.suffix.lower() not in {".xlsx", ".xls", ".csv", ".tsv"}:
            continue
        info = classify_filename(p.name)
        info["path"]   = str(p)
        info["size_kb"] = p.stat().st_size / 1024
        rows.append(info)
    return pd.DataFrame(rows)


discovered = discover_files(INPUT_DIR)
print(f"Найдено файлов: {len(discovered)}")
display_cols = ["name", "kind", "entity", "period", "granularity", "size_kb"]
discovered[display_cols] if len(discovered) else "Нет файлов"


## 6. Универсальный загрузчик данных

Файл может быть:

- **CSV / TSV** — проверяем разделитель и десятичную запятую.
- **XLSX (одна таблица)** — самый простой случай; первый столбец — время, два следующих — `P` и `Q`.
- **XLSX (многолистовой, экспорт Энерготестера)** — нужно перебрать листы и склеить нужные.
- **HTML, переименованный в .xls** — типичный экспорт российских приборов учёта (Энерготестер ПКЭ-А, ПКК-57). Внутри файла — HTML с `<table>`-тегами, расширение `.xls` для удобства открытия в Excel.
- **XML SpreadsheetML 2003** — реже, но встречается.

Загрузчик:

1. **Определяет реальный формат** файла по магическим байтам, а не по расширению.
2. Применяет соответствующий парсер: `xlrd` для BIFF, `openpyxl` для XLSX, `pd.read_html` для HTML/XML.
3. Перебирает листы / таблицы и для каждой пытается выделить колонки `(timestamp, P, Q)`.

Все числа в столбцах P / Q приводятся через `_coerce_numeric`, который понимает запятую как разделитель десятичной части и неразрывные пробелы как разделители тысяч.

In [ ]:
def _coerce_numeric(s: pd.Series) -> pd.Series:
    """Привести Series к float, обрабатывая русские запятые и пробелы.

    Все значения, которые не парсятся в число (заголовки, текст), становятся NaN.
    """
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    cleaned = (
        s.astype(str)
         .str.replace("\u00a0", "", regex=False)   # неразрывный пробел
         .str.replace(" ", "", regex=False)
         .str.replace(",", ".", regex=False)
         .replace({"": np.nan, "nan": np.nan, "None": np.nan, "-": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce")


def _parse_datetime(s: pd.Series) -> pd.DatetimeIndex:
    """Парсер дат с поддержкой интервальной метки '01.08.2023 00:00 - 01.08.2023 01:00'."""
    cleaned = s.astype(str).str.split(" - ").str[0]
    parsed = pd.to_datetime(cleaned, dayfirst=True, errors="coerce")
    if parsed.isna().mean() > 0.5:                  # видимо, ISO формат
        parsed = pd.to_datetime(cleaned, errors="coerce")
    return pd.DatetimeIndex(parsed)


def detect_actual_format(path) -> str:
    """Определяет реальный формат файла по первым байтам, игнорируя расширение.

    Возвращает одно из:
      'xls_biff'        - настоящий бинарный xls (OLE2/CFB)
      'xlsx'            - xlsx (zip-контейнер)
      'html'            - HTML, переименованный в xls (типично для приборов учёта)
      'xml_spreadsheet' - XML SpreadsheetML 2003 (реже)
      'xml_other'       - произвольный XML
      'mhtml'           - MHTML web archive
      'csv_text'        - текст / CSV без бинарной сигнатуры
      'unknown'         - не распознан
    """
    path = Path(path)
    try:
        with open(path, "rb") as f:
            header = f.read(4096)
    except Exception:
        return "unknown"

    # Бинарные сигнатуры
    if header[:8] == b"\xD0\xCF\x11\xE0\xA1\xB1\x1A\xE1":
        return "xls_biff"
    if header[:4] == b"PK\x03\x04":
        return "xlsx"

    # Снять BOM, если есть
    if header.startswith(b"\xef\xbb\xbf"):
        header = header[3:]

    # Текстовые форматы
    head_lower = header.decode("utf-8", errors="ignore").lower().lstrip()

    if head_lower.startswith("mime-version:") or "content-type: multipart/related" in head_lower[:500]:
        return "mhtml"
    if head_lower.startswith("<?xml"):
        if ("urn:schemas-microsoft-com:office:spreadsheet" in head_lower[:2000]
            or "<workbook" in head_lower[:2000]):
            return "xml_spreadsheet"
        return "xml_other"
    if head_lower.startswith(("<html", "<!doctype")) or "<html" in head_lower[:2000]:
        return "html"
    if "<table" in head_lower[:2000]:                  # фрагмент HTML без обёртки
        return "html"
    # Не нашли явных меток — допустим, текст / CSV
    return "csv_text" if head_lower else "unknown"


def _parse_spreadsheetml_2003(path) -> dict:
    """Парсер SpreadsheetML 2003 XML (старый XML-формат Microsoft Office).

    Файл выглядит как:
        <?xml version="1.0"?>
        <?mso-application progid="Excel.Sheet"?>
        <Workbook xmlns="urn:schemas-microsoft-com:office:spreadsheet">
          <Worksheet ss:Name="Лист1">
            <Table>
              <Row>
                <Cell><Data ss:Type="String">Время</Data></Cell>
                <Cell><Data ss:Type="Number">2730.5</Data></Cell>
              </Row>
              ...

    Возвращает {sheet_name: DataFrame}.
    """
    import xml.etree.ElementTree as ET

    NS = "urn:schemas-microsoft-com:office:spreadsheet"

    try:
        tree = ET.parse(str(path))
    except ET.ParseError as exc:
        # Попробуем восстановить из файла, у которого может быть BOM
        with open(path, "rb") as f:
            content = f.read()
        if content.startswith(b"\xef\xbb\xbf"):
            content = content[3:]
        tree = ET.ElementTree(ET.fromstring(content))

    root = tree.getroot()

    result = {}
    worksheets = root.findall(f"{{{NS}}}Worksheet")
    for ws_idx, ws in enumerate(worksheets):
        sheet_name = ws.get(f"{{{NS}}}Name") or f"Sheet{ws_idx+1}"
        table = ws.find(f"{{{NS}}}Table")
        if table is None:
            continue

        rows_data = []
        max_cols = 0
        for row in table.findall(f"{{{NS}}}Row"):
            row_data = []
            col_idx = 0
            for cell in row.findall(f"{{{NS}}}Cell"):
                # ss:Index указывает абсолютную позицию (для разреженных данных)
                idx_attr = cell.get(f"{{{NS}}}Index")
                if idx_attr:
                    try:
                        target_idx = int(idx_attr) - 1
                        while col_idx < target_idx:
                            row_data.append(None)
                            col_idx += 1
                    except ValueError:
                        pass

                data_elem = cell.find(f"{{{NS}}}Data")
                if data_elem is not None:
                    val = data_elem.text
                    dtype = data_elem.get(f"{{{NS}}}Type")
                    if dtype == "Number" and val is not None:
                        try:
                            val = float(val)
                        except ValueError:
                            pass
                    elif dtype == "Boolean" and val is not None:
                        val = val == "1"
                else:
                    val = None

                row_data.append(val)
                col_idx += 1

                # ss:MergeAcross расширяет ячейку вправо
                merge_across = cell.get(f"{{{NS}}}MergeAcross")
                if merge_across:
                    try:
                        for _ in range(int(merge_across)):
                            row_data.append(None)
                            col_idx += 1
                    except ValueError:
                        pass

            rows_data.append(row_data)
            if len(row_data) > max_cols:
                max_cols = len(row_data)

        # Выравниваем длину строк
        for r in rows_data:
            while len(r) < max_cols:
                r.append(None)

        if rows_data:
            result[sheet_name] = pd.DataFrame(rows_data)

    return result


def _read_excel_all_sheets(path) -> dict:
    """Читает все листы / таблицы файла, независимо от его реального формата.

    Возвращает {sheet_name: DataFrame без заголовка}.
    """
    path = Path(path)
    suffix = path.suffix.lower()

    # Для CSV/TSV не определяем формат — сразу читаем
    if suffix in {".csv", ".tsv"}:
        sep = "\t" if suffix == ".tsv" else ","
        try:
            return {"sheet1": pd.read_csv(path, sep=sep, header=None)}
        except Exception:
            return {"sheet1": pd.read_csv(path, sep=None, engine="python", header=None)}

    fmt = detect_actual_format(path)

    # Подготовим список попыток в правильном порядке
    attempts = []

    if fmt == "xls_biff":
        attempts.append(("xlrd[biff]",
            lambda: pd.read_excel(path, sheet_name=None, engine="xlrd", header=None)))
    elif fmt == "xlsx":
        attempts.append(("openpyxl",
            lambda: pd.read_excel(path, sheet_name=None, engine="openpyxl", header=None)))
    elif fmt == "xml_spreadsheet":
        # Это формат большинства приборных экспортов — отдельный парсер
        attempts.append(("spreadsheetml_2003", lambda: _parse_spreadsheetml_2003(path)))
    elif fmt in ("html", "xml_other", "mhtml"):
        attempts.append(("read_html",
            lambda: {f"table_{i}": t for i, t in enumerate(pd.read_html(path, header=None))}))

    # Универсальные fallbacks (на случай, если детектор ошибся)
    if suffix == ".xlsx":
        attempts.append(("openpyxl",
            lambda: pd.read_excel(path, sheet_name=None, engine="openpyxl", header=None)))
    if suffix == ".xls":
        # SpreadsheetML 2003 — типичный формат для Энерготестера
        attempts.append(("spreadsheetml_2003", lambda: _parse_spreadsheetml_2003(path)))
        # Реальный BIFF
        attempts.append(("xlrd",
            lambda: pd.read_excel(path, sheet_name=None, engine="xlrd", header=None)))
        # HTML с расширением .xls — тоже встречается
        attempts.append(("read_html_disguised",
            lambda: {f"table_{i}": t for i, t in enumerate(pd.read_html(path, header=None))}))
        # XLSX с расширением .xls — иногда экспортёры так делают
        attempts.append(("openpyxl_disguised",
            lambda: pd.read_excel(path, sheet_name=None, engine="openpyxl", header=None)))

    # Самый последний шанс
    attempts.append(("read_html_lastchance",
        lambda: {f"table_{i}": t for i, t in enumerate(pd.read_html(path, header=None))}))

    last_err = None
    for name, attempt in attempts:
        try:
            res = attempt()
            if res:
                return res
        except Exception as exc:
            last_err = (name, str(exc)[:120])
            continue

    raise ValueError(f"Не удалось прочитать {path.name} "
                     f"(детект: {fmt}, последняя ошибка: {last_err})")


def _read_csv(path) -> pd.DataFrame:
    sep = "\t" if Path(path).suffix.lower() == ".tsv" else ","
    return pd.read_csv(path, sep=sep, header=None)


def _try_load_pq_table(df_raw: pd.DataFrame):
    """Пытается интерпретировать таблицу как (datetime, P, Q [, ...]).

    Стратегия:
    1. Перебираем кандидатов на колонку с временной меткой (первые 5 столбцов).
    2. В каждой колонке находим **самый длинный непрерывный блок** дат,
       попадающих в диапазон 1990-2100. Это нужно, чтобы:
       - отфильтровать ложные срабатывания на числах вида '2 730,5'
         (которые pd.to_datetime может прочитать как '730-02-01'),
       - игнорировать одиночные «даты» в шапках типа 'Период: 15.07.2024…'.
    3. Берём P и Q из следующих двух столбцов. Заголовки и текстовые ячейки
       при конвертации в число становятся NaN и фильтруются.

    Возвращает None, если не удалось.
    """
    if df_raw is None or df_raw.empty or df_raw.shape[1] < 3:
        return None

    n_cols_to_try = min(5, df_raw.shape[1])
    best = None  # (block_start_pos, dt_col, block_length)

    for dt_col in range(n_cols_to_try):
        col = df_raw.iloc[:, dt_col]
        parsed = pd.to_datetime(col.astype(str).str.split(" - ").str[0],
                                dayfirst=True, errors="coerce")
        # фильтр диапазона года, отсекающий числовые ложные срабатывания
        try:
            yrs = parsed.dt.year
            valid_mask = parsed.notna() & yrs.ge(1990) & yrs.le(2100)
        except Exception:
            continue

        if int(valid_mask.sum()) < 5:
            continue

        # Поиск самого длинного непрерывного True-блока
        best_start, best_len = None, 0
        cur_start, cur_len = None, 0
        for idx, v in enumerate(valid_mask.values):
            if v:
                if cur_start is None:
                    cur_start, cur_len = idx, 1
                else:
                    cur_len += 1
            else:
                if cur_len > best_len:
                    best_start, best_len = cur_start, cur_len
                cur_start, cur_len = None, 0
        if cur_len > best_len:
            best_start, best_len = cur_start, cur_len

        if best_len < 5:
            continue

        if best is None or best_len > best[2]:
            best = (best_start, dt_col, best_len)

    if best is None:
        return None

    data_start, dt_col, _ = best
    if df_raw.shape[1] < dt_col + 3:
        return None

    body = df_raw.iloc[data_start:].reset_index(drop=True)
    parsed_idx = _parse_datetime(body.iloc[:, dt_col])
    # Важно: присваиваем через .values, иначе pandas попытается выровнять по индексу
    # (у body — integer-индекс, у out — DatetimeIndex) и поставит NaN везде.
    out = pd.DataFrame(index=parsed_idx)
    out["P"] = _coerce_numeric(body.iloc[:, dt_col + 1]).values
    out["Q"] = _coerce_numeric(body.iloc[:, dt_col + 2]).values

    out = out[~out.index.isna()].sort_index()
    out = out.dropna(subset=["P", "Q"], how="all")
    if out.empty or out["P"].dropna().empty:
        return None

    out["S"]  = np.sqrt(out["P"]**2 + out["Q"]**2)
    out["pf"] = out["P"] / out["S"].replace(0, np.nan)
    return out[["P", "Q", "S", "pf"]]


def load_pq_file(path):
    """Универсальная загрузка файла электропотребления.

    Перебирает все листы / таблицы файла и возвращает первый успешный результат.
    """
    path = Path(path)
    try:
        sheets = _read_excel_all_sheets(path)
        for name, df_raw in sheets.items():
            res = _try_load_pq_table(df_raw)
            if res is not None and len(res) > 5:
                res.attrs["sheet"] = name
                return res
        return None
    except Exception as exc:
        print(f"   [load_pq_file] ошибка для {path.name}: {exc}")
        return None


## 7. Загрузка всех найденных файлов

Создаём словарь `loaded` с ключом `(entity, period)` для удобного обращения.

In [ ]:
loaded = {}
load_log = []

for _, row in discovered.iterrows():
    if row["kind"] in ("ore_throughput", "mill_measurement", "unknown",
                       "bus_monthly_combined"):
        # эти типы обрабатываются в специальных блоках ниже
        continue

    df = load_pq_file(Path(row["path"]))
    if df is None or df.empty:
        load_log.append({**row, "loaded": False, "n_rows": 0})
        print(f"[skip] {row['name']}")
        continue

    key = (row["entity"], row["period"], row["granularity"])
    loaded[key] = df
    load_log.append({**row, "loaded": True, "n_rows": len(df),
                     "t_start": str(df.index.min()), "t_end": str(df.index.max())})
    print(f"[OK]   {row['entity']:<12s} | {row['period']:<10s} | "
          f"{row['granularity']:<8s} | n={len(df):>6d} | "
          f"{df.index.min()} … {df.index.max()}")

pd.DataFrame(load_log).to_csv(OUTPUT_DIR / "tables/load_log.csv", index=False,
                              encoding="utf-8-sig")
print(f"\nЗагружено источников: {len(loaded)}")


## 8. Описательная статистика, тесты согласия и стационарности

Для каждого источника считаем:

- среднее, стандартное отклонение, медиану, 5/95-перцентили, минимум, максимум, асимметрию, эксцесс, коэффициент вариации;
- доверительный интервал среднего по t-распределению (α = 0,05);
- тесты согласия с нормальным распределением: Шапиро–Уилк, Колмогоров–Смирнов, Андерсон–Дарлинг, D'Агостино–Пирсон;
- тесты стационарности: ADF, KPSS.

Все таблицы сохраняются в `OUTPUT_DIR/tables/`.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

def basic_stats(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in ("P", "Q", "S", "pf"):
        s = df[col].dropna()
        if s.empty:
            continue
        rows.append({
            "var":   col,
            "n":     int(len(s)),
            "mean":  s.mean(),
            "std":   s.std(ddof=1),
            "median": s.median(),
            "p05":   s.quantile(0.05),
            "p95":   s.quantile(0.95),
            "min":   s.min(),
            "max":   s.max(),
            "skew":  stats.skew(s),
            "kurt":  stats.kurtosis(s),
            "cv":    s.std(ddof=1) / s.mean() if abs(s.mean()) > 1e-9 else np.nan,
        })
    return pd.DataFrame(rows).set_index("var")


def confidence_interval(s: pd.Series, alpha: float = 0.05):
    s = s.dropna()
    n = len(s)
    if n < 2:
        return np.nan, np.nan, np.nan
    mean = s.mean()
    se   = s.std(ddof=1) / np.sqrt(n)
    t    = stats.t.ppf(1 - alpha/2, df=n-1)
    return mean - t*se, mean, mean + t*se


def goodness_of_fit_tests(s: pd.Series) -> pd.DataFrame:
    s = s.dropna().values
    out = []
    if len(s) < 8:
        return pd.DataFrame(out)
    try:
        ss = s if len(s) <= 5000 else np.random.choice(s, 5000, replace=False)
        sw = stats.shapiro(ss)
        out.append({"test": "Shapiro-Wilk", "stat": sw.statistic, "pvalue": sw.pvalue})
    except Exception as exc:
        out.append({"test": "Shapiro-Wilk", "err": str(exc)})
    try:
        ks = stats.kstest((s - s.mean()) / s.std(ddof=1), "norm")
        out.append({"test": "Kolmogorov-Smirnov", "stat": ks.statistic, "pvalue": ks.pvalue})
    except Exception as exc:
        out.append({"test": "Kolmogorov-Smirnov", "err": str(exc)})
    try:
        ad = stats.anderson(s, dist="norm")
        out.append({"test": "Anderson-Darling", "stat": ad.statistic,
                    "pvalue_5pct": ad.significance_level[2]})
    except Exception as exc:
        out.append({"test": "Anderson-Darling", "err": str(exc)})
    try:
        d = stats.normaltest(s)
        out.append({"test": "DAgostino-Pearson", "stat": d.statistic, "pvalue": d.pvalue})
    except Exception as exc:
        out.append({"test": "DAgostino-Pearson", "err": str(exc)})
    return pd.DataFrame(out)


def stationarity_tests(s: pd.Series) -> pd.DataFrame:
    s = s.dropna()
    rows = []
    try:
        a = adfuller(s, autolag="AIC")
        rows.append({"test": "ADF", "stat": a[0], "pvalue": a[1],
                     "crit_5pct": a[4]["5%"], "stationary": a[1] < 0.05})
    except Exception as exc:
        rows.append({"test": "ADF", "err": str(exc)})
    try:
        k = kpss(s, regression="c", nlags="auto")
        rows.append({"test": "KPSS", "stat": k[0], "pvalue": k[1],
                     "crit_5pct": k[3]["5%"], "stationary": k[1] >= 0.05})
    except Exception as exc:
        rows.append({"test": "KPSS", "err": str(exc)})
    return pd.DataFrame(rows)


# Прогоним по всем источникам
summary_rows = []
for (ent, per, gran), df in loaded.items():
    label = f"{ent}_{per}_{gran}"
    bs = basic_stats(df)
    bs.to_csv(OUTPUT_DIR / f"tables/stats_{label}.csv", float_format="%.4f", encoding="utf-8-sig")
    gof = goodness_of_fit_tests(df["P"])
    gof.to_csv(OUTPUT_DIR / f"tables/gof_{label}.csv", index=False, encoding="utf-8-sig")
    st = stationarity_tests(df["P"])
    st.to_csv(OUTPUT_DIR / f"tables/stationarity_{label}.csv", index=False, encoding="utf-8-sig")

    ci_lo, ci_mid, ci_hi = confidence_interval(df["P"])
    summary_rows.append({
        "entity": ent, "period": per, "granularity": gran,
        "n": int(len(df)),
        "P_mean": df["P"].mean(), "P_std": df["P"].std(),
        "P_CI_low": ci_lo, "P_CI_high": ci_hi,
        "Q_mean": df["Q"].mean(), "Q_std": df["Q"].std(),
        "pf_mean": df["pf"].mean(),
        "P_skew": bs.loc["P", "skew"], "P_kurt": bs.loc["P", "kurt"],
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / "tables/summary_all_sources.csv",
               index=False, float_format="%.4f", encoding="utf-8-sig")
print("Сводная таблица:")
summary


## 9. Визуализации EDA

Для каждого источника генерируются:

- временные ряды P (вверху) и Q (внизу),
- гистограммы P, Q, S, pf с наложенной нормальной кривой,
- boxplot по сезонам (зима / весна / лето / осень),
- boxplot по часам суток (для почасовых рядов),
- STL-декомпозиция P (тренд / сезонность / остаток),
- ACF и PACF P.

Все графики — в `OUTPUT_DIR/figures/`.

In [ ]:
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

FIG_DIR = OUTPUT_DIR / "figures"


def _save(fig, name):
    fig.savefig(FIG_DIR / name)
    plt.close(fig)


def plot_timeseries(df, label):
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    axes[0].plot(df.index, df["P"], color="#1f77b4", lw=1.0)
    axes[0].set_ylabel("P, kW"); axes[0].set_title(label); axes[0].grid(alpha=0.3)
    axes[1].plot(df.index, df["Q"], color="#d62728", lw=1.0)
    axes[1].set_ylabel("Q, kvar"); axes[1].grid(alpha=0.3)
    axes[1].xaxis.set_major_locator(mdates.AutoDateLocator())
    axes[1].xaxis.set_major_formatter(
        mdates.ConciseDateFormatter(axes[1].xaxis.get_major_locator()))
    fig.tight_layout()
    _save(fig, f"ts_{label}.png")


def plot_distributions(df, label):
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    for ax, col, color in zip(axes.flat, ["P", "Q", "S", "pf"],
                              ["#1f77b4", "#d62728", "#2ca02c", "#9467bd"]):
        s = df[col].dropna()
        if len(s) < 5:
            continue
        ax.hist(s, bins=40, density=True, alpha=0.7, color=color, edgecolor="white")
        xs = np.linspace(s.min(), s.max(), 200)
        ax.plot(xs, stats.norm.pdf(xs, s.mean(), s.std(ddof=1)), "k--", lw=1.2)
        ax.set_title(col); ax.grid(alpha=0.3)
    fig.suptitle(label); fig.tight_layout()
    _save(fig, f"hist_{label}.png")


def plot_boxplot_season(df, label):
    season_map = {12:"Winter",1:"Winter",2:"Winter",
                  3:"Spring",4:"Spring",5:"Spring",
                  6:"Summer",7:"Summer",8:"Summer",
                  9:"Autumn",10:"Autumn",11:"Autumn"}
    tmp = df.copy()
    tmp["season"] = tmp.index.month.map(season_map)
    order = ["Winter","Spring","Summer","Autumn"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    sns.boxplot(data=tmp, x="season", y="P", order=order, ax=axes[0], palette="Blues")
    sns.boxplot(data=tmp, x="season", y="Q", order=order, ax=axes[1], palette="Reds")
    axes[0].set_title(f"{label} | Active power"); axes[1].set_title(f"{label} | Reactive power")
    for a in axes: a.grid(alpha=0.3)
    fig.tight_layout()
    _save(fig, f"box_season_{label}.png")


def plot_boxplot_hour(df, label):
    if df.attrs.get("granularity") != "hourly" and (df.index[1]-df.index[0]) > pd.Timedelta(hours=2):
        return
    tmp = df.copy(); tmp["hour"] = tmp.index.hour
    fig, ax = plt.subplots(figsize=(11, 4.5))
    sns.boxplot(data=tmp, x="hour", y="P", ax=ax)
    ax.set_title(f"{label} | Hourly distribution of P"); ax.grid(alpha=0.3)
    fig.tight_layout(); _save(fig, f"box_hour_{label}.png")


def plot_stl(df, label, period):
    s = df["P"].dropna()
    if len(s) < 2*period + 5:
        return
    stl = STL(s, period=period, robust=True).fit()
    fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
    axes[0].plot(s.index, s.values, "k", lw=0.8); axes[0].set_ylabel("Observed")
    axes[1].plot(stl.trend.index, stl.trend, color="#1f77b4");      axes[1].set_ylabel("Trend")
    axes[2].plot(stl.seasonal.index, stl.seasonal, color="#2ca02c");axes[2].set_ylabel("Seasonal")
    axes[3].plot(stl.resid.index, stl.resid, color="#d62728");      axes[3].set_ylabel("Residual")
    for a in axes: a.grid(alpha=0.3)
    fig.suptitle(label); fig.tight_layout()
    _save(fig, f"stl_{label}.png")


def plot_acf_pacf(df, label):
    s = df["P"].dropna()
    lags = min(48, len(s)//2 - 1) if len(s) > 100 else min(24, len(s)//2 - 1)
    if lags < 5:
        return
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    plot_acf(s, lags=lags, ax=axes[0]); axes[0].set_title("ACF")
    plot_pacf(s, lags=lags, ax=axes[1], method="ywm"); axes[1].set_title("PACF")
    fig.suptitle(label); fig.tight_layout()
    _save(fig, f"acf_pacf_{label}.png")


for (ent, per, gran), df in loaded.items():
    label = f"{ent}_{per}_{gran}"
    print(f"plotting {label} ...")
    plot_timeseries(df, label)
    plot_distributions(df, label)
    plot_boxplot_season(df, label)
    plot_boxplot_hour(df, label)
    period = 12 if gran == "monthly" else 24
    plot_stl(df, label, period)
    plot_acf_pacf(df, label)

print("\nГрафики EDA готовы.")


## 10. Кросс-секционный анализ: панели и удельные веса

Собираем широкую таблицу по всем источникам одной гранулярности и считаем:

- корреляционные матрицы P / Q между источниками,
- удельные веса каждого источника в суммарной мощности предприятия,
- сводную таблицу средних, минимумов, максимумов.

In [ ]:
def assemble_panel(loaded_dict, granularity, var):
    """Собрать панель: время × источник."""
    sers = []
    for (ent, per, gran), df in loaded_dict.items():
        if gran != granularity:
            continue
        col = f"{ent}_{per}"
        sers.append(df[var].rename(col))
    if not sers:
        return pd.DataFrame()
    return pd.concat(sers, axis=1).sort_index()


def plot_corr(panel, title, fname):
    if panel.empty or panel.shape[1] < 2:
        return
    corr = panel.corr(method="pearson")
    fig, ax = plt.subplots(figsize=(min(12, max(7, 0.6*len(corr))),
                                    min(10, max(5, 0.5*len(corr)))))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
                square=True, cbar_kws={"shrink": 0.7}, ax=ax)
    ax.set_title(title); fig.tight_layout()
    fig.savefig(FIG_DIR / fname); plt.close(fig)


def plot_share_pie(panel, title, fname):
    if panel.empty:
        return
    means = panel.mean()
    means = means[means > 0]
    if means.empty:
        return
    fig, ax = plt.subplots(figsize=(7.5, 7.5))
    ax.pie(means.values, labels=means.index, autopct="%1.1f%%", startangle=90,
           wedgeprops=dict(edgecolor="white", linewidth=1.2),
           textprops={"fontsize": 10})
    ax.set_title(title); fig.tight_layout()
    fig.savefig(FIG_DIR / fname); plt.close(fig)


for gran in ("monthly", "hourly"):
    pp = assemble_panel(loaded, gran, "P")
    qp = assemble_panel(loaded, gran, "Q")
    if pp.empty:
        continue
    pp.to_csv(OUTPUT_DIR / f"tables/panel_P_{gran}.csv", encoding="utf-8-sig")
    qp.to_csv(OUTPUT_DIR / f"tables/panel_Q_{gran}.csv", encoding="utf-8-sig")
    plot_corr(pp, f"Pearson correlations of P ({gran})", f"corr_P_{gran}.png")
    plot_corr(qp, f"Pearson correlations of Q ({gran})", f"corr_Q_{gran}.png")
    plot_share_pie(pp, f"Share of mean P by source ({gran})", f"pie_P_{gran}.png")
    plot_share_pie(qp, f"Share of mean Q by source ({gran})", f"pie_Q_{gran}.png")

print("Кросс-секционный анализ выполнен.")


## 11. Анализ инструментальных измерений мельниц

Файлы вида `15.07.2024 10.21.31-15.07.2024 11.11.13.xlsx` — это экспорты приборов учёта Энерготестер ПКЭ-А (или подобных). Все они имеют одинаковую внутреннюю структуру с **8 листами**: «Токи и напряжения», **«Мощности»**, «Углы», «ПКЭ», «Гармоники тока», «Гармоники лин. напряжения», «Интергармоники тока», «Интергармоники лин. напряжения».

Универсальный загрузчик секции 6 не подходит для этих файлов по двум причинам:

1. **P, Q находятся не на первом листе**, а на отдельном листе «Мощности».
2. **В колонке «Время» часто только `HH:MM:SS` без даты** — pandas подставляет туда сегодняшнюю дату или 1900-01-01. Поэтому корректные даты приходится извлекать из имени файла через regex.

Ниже — специализированный парсер `load_mill_session()`, учитывающий обе особенности. Дополнительно `load_mill_pqe()` извлекает с листа «ПКЭ» интегральные показатели качества электроэнергии (U₁, U₂, U₀, K_2U, K_0U, I₁, P₁, и т.д.) — это даёт нам напрямую значения для оценки cosφ и несимметрии.

In [ ]:
import re

# Имена листов в приборных файлах. Список приоритета — берём первое совпадение.
_MILL_POWER_SHEETS = ["Мощности", "Powers", "Power", "P_Q", "PQ"]
_MILL_PQE_SHEETS   = ["ПКЭ", "PQE", "Quality", "Power Quality"]

# Паттерны имён колонок P и Q. Регистронезависимо.
_P_PATTERNS = [
    r"^P[\s,]*Σ",          # P, Σ Вт
    r"^P\s*\(?\s*Σ",      # P (Σ
    r"^P\s*суммарн",
    r"^P\s*total",
    r"^Pa?[\s,]*Σ",
    r"^P[\s,]*Вт",          # P, Вт
    r"^P[\s,]*W",
    r"^P[\s,]*кВт",
    r"^P[\s,]*kW",
    r"^P\b",                # просто 'P' как точное совпадение
    r"P_total",
    r"Активная.*мощность",
]
_Q_PATTERNS = [
    r"^Q[\s,]*Σ",
    r"^Q\s*\(?\s*Σ",
    r"^Q\s*суммарн",
    r"^Q\s*total",
    r"^Q[\s,]*вар",
    r"^Q[\s,]*var",
    r"^Q[\s,]*квар",
    r"^Q[\s,]*kvar",
    r"^Q\b",
    r"Q_total",
    r"Реактивная.*мощность",
]


def _extract_session_dates(filename: str) -> tuple[pd.Timestamp | None, pd.Timestamp | None]:
    """Извлекает начальную и конечную дату-время сессии из имени файла.

    Поддерживает форматы:
      "15.07.2024 10.21.31-15.07.2024 11.11.13.xlsx"
      "16.07.2024 06.28.31-16.07.2024 13.30.05.xlsx"
      "160724_170724.xlsx" -> (16.07.2024 00:00, 17.07.2024 23:59)
    """
    name = Path(filename).stem
    # Полные даты с временем "ДД.ММ.ГГГГ ЧЧ.ММ.СС"
    pat_full = re.compile(
        r"(\d{2})\.(\d{2})\.(\d{4})\s+(\d{2})\.(\d{2})\.(\d{2})"
    )
    matches = pat_full.findall(name)
    if len(matches) >= 2:
        d1, m1, y1, h1, mi1, s1 = matches[0]
        d2, m2, y2, h2, mi2, s2 = matches[1]
        try:
            t1 = pd.Timestamp(int(y1), int(m1), int(d1), int(h1), int(mi1), int(s1))
            t2 = pd.Timestamp(int(y2), int(m2), int(d2), int(h2), int(mi2), int(s2))
            return t1, t2
        except (ValueError, TypeError):
            pass
    if len(matches) == 1:
        d, m, y, h, mi, s = matches[0]
        try:
            t1 = pd.Timestamp(int(y), int(m), int(d), int(h), int(mi), int(s))
            return t1, None
        except (ValueError, TypeError):
            pass
    # Формат "ДДММГГ_ДДММГГ"
    pat_short = re.compile(r"(\d{2})(\d{2})(\d{2})_(\d{2})(\d{2})(\d{2})")
    m = pat_short.search(name)
    if m:
        d1, mo1, y1, d2, mo2, y2 = m.groups()
        try:
            t1 = pd.Timestamp(2000 + int(y1), int(mo1), int(d1), 0, 0, 0)
            t2 = pd.Timestamp(2000 + int(y2), int(mo2), int(d2), 23, 59, 59)
            return t1, t2
        except (ValueError, TypeError):
            pass
    return None, None


def _find_sheet_by_names(sheets: dict, candidates: list[str]) -> tuple[str, pd.DataFrame] | None:
    """Ищет лист в `sheets` по списку имён (case-insensitive). Возвращает (name, df) или None."""
    for cand in candidates:
        for actual_name in sheets.keys():
            if str(actual_name).strip().lower() == cand.strip().lower():
                return actual_name, sheets[actual_name]
    return None


def _find_header_row(df_raw: pd.DataFrame, max_rows: int = 8) -> int:
    """Определяет номер строки заголовка: первая строка, в которой
    в первой ячейке стоит «Время», «Time», или дата."""
    for i in range(min(max_rows, len(df_raw))):
        v = df_raw.iat[i, 0]
        s = str(v).strip().lower() if v is not None else ""
        if s in ("время", "time", "timestamp", "дата", "date"):
            return i
    return 0  # если шапка не найдена, считаем что данные с первой строки


def _find_pq_columns(headers: list, p_pats: list, q_pats: list) -> tuple[int | None, int | None]:
    """Перебирает шапку и возвращает индексы колонок P и Q по regex-паттернам."""
    p_idx, q_idx = None, None
    for j, h in enumerate(headers):
        if h is None:
            continue
        s = str(h).strip()
        if p_idx is None:
            for pat in p_pats:
                if re.search(pat, s, re.IGNORECASE):
                    p_idx = j
                    break
        if q_idx is None:
            for pat in q_pats:
                if re.search(pat, s, re.IGNORECASE):
                    q_idx = j
                    break
    return p_idx, q_idx


def _build_full_datetimes(time_col: pd.Series, base_date: pd.Timestamp | None) -> pd.DatetimeIndex:
    """Строит полные даты-время.

    Если `base_date` задан (известна сессия из имени файла), мы знаем что все
    измерения должны быть в окрестности этой даты. В этом случае:
      - Если в ячейке только HH:MM[:SS] -> комбинируем с base_date
      - Если в ячейке полная дата в окрестности base_date (±2 дня) -> используем её
      - Если pandas подставил сегодняшнюю дату или 1900-01-01 (фейковая интерпретация
        time-only ячейки) -> берём только time-of-day и комбинируем с base_date
    Корректирует переход через полночь.
    """
    out = []
    last_full = base_date
    today = pd.Timestamp.now().normalize()
    for v in time_col:
        ts = pd.NaT

        # Извлекаем (час, минута, секунда) из любого формата
        time_only = None
        full_date = None

        if isinstance(v, pd.Timestamp):
            time_only = (v.hour, v.minute, v.second)
            if 1990 <= v.year <= 2100:
                full_date = v
        elif hasattr(v, "hour") and hasattr(v, "minute"):
            # datetime.time объект
            time_only = (v.hour, v.minute, getattr(v, "second", 0))
        elif isinstance(v, (int, float)) and not pd.isna(v):
            # Excel-сериал даты: целая часть = дни от 30.12.1899, дробная = время
            try:
                full_ts = pd.Timestamp("1899-12-30") + pd.Timedelta(days=float(v))
                if 1990 <= full_ts.year <= 2100:
                    full_date = full_ts
                    time_only = (full_ts.hour, full_ts.minute, full_ts.second)
                else:
                    # значит это просто time-of-day как fraction of day
                    frac = float(v) % 1.0
                    h = int(frac * 24)
                    mi = int((frac * 24 - h) * 60)
                    se = int(((frac * 24 - h) * 60 - mi) * 60)
                    time_only = (h, mi, se)
            except (ValueError, OverflowError):
                pass
        elif v is not None:
            s = str(v).strip()
            try_full = pd.to_datetime(s, dayfirst=True, errors="coerce")
            if pd.notna(try_full) and 1990 <= try_full.year <= 2100:
                full_date = try_full
                time_only = (try_full.hour, try_full.minute, try_full.second)
            else:
                # только время HH:MM или HH:MM:SS
                m = re.match(r"^(\d{1,2})[:.](\d{2})(?:[:.](\d{2}))?$", s)
                if m:
                    time_only = (int(m.group(1)), int(m.group(2)), int(m.group(3) or 0))

        # Логика выбора окончательной timestamp:
        if base_date is not None:
            # Известна дата сессии — используем её для подстановки
            if full_date is not None:
                # Проверяем: если full_date — это сегодняшняя дата (фейк от pandas)
                # И base_date далеко от сегодня — значит ячейка содержала только время
                gap_to_today = abs((full_date.normalize() - today).days)
                gap_to_base  = abs((full_date.normalize() - base_date.normalize()).days)
                if gap_to_today <= 1 and gap_to_base > 7:
                    # Это сегодняшняя дата от pandas, а сессия далеко — берём только время
                    if time_only is not None:
                        ts = pd.Timestamp(base_date.year, base_date.month, base_date.day,
                                           *time_only)
                elif gap_to_base <= 3:
                    # full_date в окрестности сессии — используем как есть
                    ts = full_date
                else:
                    # full_date далеко от сессии — используем только время
                    if time_only is not None:
                        ts = pd.Timestamp(base_date.year, base_date.month, base_date.day,
                                           *time_only)
            elif time_only is not None:
                ts = pd.Timestamp(base_date.year, base_date.month, base_date.day,
                                   *time_only)
        else:
            # Дата сессии неизвестна — используем full_date если есть
            if full_date is not None:
                ts = full_date

        # Корректировка перехода через полночь
        if pd.notna(ts) and last_full is not None and ts < last_full - pd.Timedelta(hours=12):
            ts = ts + pd.Timedelta(days=1)
        if pd.notna(ts):
            last_full = ts
        out.append(ts)
    return pd.DatetimeIndex(out)


def load_mill_session(path) -> pd.DataFrame | None:
    """Загружает одну сессию измерения мельницы из xlsx-файла прибора учёта.

    Стратегия:
      1. Извлекает начальную дату-время сессии из имени файла.
      2. Открывает лист «Мощности» (или эквивалент).
      3. Находит колонки P и Q по именам шапки.
      4. Соединяет HH:MM:SS из ячейки с датой из имени.
      5. Конвертирует Вт → кВт и вар → квар, если значения выглядят как ватты.
    """
    p = Path(path)
    base_t1, base_t2 = _extract_session_dates(p.name)

    try:
        sheets = _read_excel_all_sheets(p)
    except Exception as exc:
        print(f"   [ERR] {p.name}: {exc}")
        return None

    found = _find_sheet_by_names(sheets, _MILL_POWER_SHEETS)
    if found is None:
        print(f"   [WARN] {p.name}: лист 'Мощности' не найден; "
              f"доступны {list(sheets.keys())[:8]}")
        return None
    sheet_name, df_raw = found

    if df_raw.empty or df_raw.shape[1] < 3:
        print(f"   [WARN] {p.name}: лист '{sheet_name}' слишком мал ({df_raw.shape})")
        return None

    header_row = _find_header_row(df_raw)
    headers = df_raw.iloc[header_row].tolist()
    p_idx, q_idx = _find_pq_columns(headers, _P_PATTERNS, _Q_PATTERNS)

    if p_idx is None or q_idx is None:
        print(f"   [WARN] {p.name}: P/Q не опознаны в шапке '{sheet_name}'.")
        print(f"          headers: {[str(h)[:25] for h in headers[:15]]}")
        return None

    body = df_raw.iloc[header_row + 1:].reset_index(drop=True)
    idx = _build_full_datetimes(body.iloc[:, 0], base_t1)

    out = pd.DataFrame(index=idx)
    p_vals = _coerce_numeric(body.iloc[:, p_idx]).values
    q_vals = _coerce_numeric(body.iloc[:, q_idx]).values

    # Если значения большие (>50 000) — скорее всего ватты, переводим в кВт
    p_unit_label = str(headers[p_idx])
    q_unit_label = str(headers[q_idx])
    p_in_watts = ("Вт" in p_unit_label or " W" in p_unit_label) and "кВт" not in p_unit_label and "kW" not in p_unit_label
    q_in_var   = ("вар" in q_unit_label.lower() or " var" in q_unit_label.lower()) \
                 and "квар" not in q_unit_label.lower() and "kvar" not in q_unit_label.lower()
    if p_in_watts:
        p_vals = p_vals / 1000.0
    if q_in_var:
        q_vals = q_vals / 1000.0

    out["P"] = p_vals
    out["Q"] = q_vals
    out = out[~out.index.isna()].sort_index()
    out = out.dropna(subset=["P", "Q"], how="all")

    # Удаляем агрегатные строки (где время не валидно или мощность подозрительно мала)
    if out.empty:
        return None

    out["S"]  = np.sqrt(out["P"]**2 + out["Q"]**2)
    out["pf"] = out["P"] / out["S"].replace(0, np.nan)
    out.attrs["sheet"]   = sheet_name
    out.attrs["session"] = (base_t1, base_t2)
    out.attrs["p_header"] = p_unit_label
    out.attrs["q_header"] = q_unit_label
    return out[["P", "Q", "S", "pf"]]


def load_mill_pqe(path) -> pd.DataFrame | None:
    """Загружает интегральные показатели качества с листа 'ПКЭ'.

    Возвращает DataFrame с колонками: U_1, U_2, U_0, K_2U, K_0U,
    I_1, I_2, I_0, P_1, P_2, P_0 и т.д. — что найдено в шапке.
    """
    p = Path(path)
    base_t1, _ = _extract_session_dates(p.name)
    try:
        sheets = _read_excel_all_sheets(p)
    except Exception:
        return None

    found = _find_sheet_by_names(sheets, _MILL_PQE_SHEETS)
    if found is None:
        return None
    sheet_name, df_raw = found
    if df_raw.empty or df_raw.shape[1] < 3:
        return None

    header_row = _find_header_row(df_raw)
    headers = df_raw.iloc[header_row].tolist()
    body = df_raw.iloc[header_row + 1:].reset_index(drop=True)
    idx = _build_full_datetimes(body.iloc[:, 0], base_t1)

    out = pd.DataFrame(index=idx)
    for j, h in enumerate(headers[1:], start=1):  # пропустим колонку «Время»
        if h is None:
            continue
        col_name = re.sub(r"\s+", "_", str(h).strip())
        col_name = re.sub(r"[,()]", "", col_name)
        out[col_name] = _coerce_numeric(body.iloc[:, j]).values

    out = out[~out.index.isna()].sort_index()
    out.attrs["sheet"] = sheet_name
    return out


In [ ]:
mill_files = discovered[discovered["kind"] == "mill_measurement"]
mill_records = {}
mill_pqe_records = {}

print("=" * 78)
print("ЗАГРУЗКА ПРИБОРНЫХ ИЗМЕРЕНИЙ С ЛИСТА \"Мощности\"")
print("=" * 78)

for _, row in mill_files.iterrows():
    p = Path(row["path"])
    print(f"\n--- {p.name} ({p.stat().st_size/1024:.0f} KB) ---")

    base_t1, base_t2 = _extract_session_dates(p.name)
    if base_t1 is not None:
        print(f"   сессия: {base_t1} ... {base_t2 or '?'}")

    df = load_mill_session(p)
    if df is None or df.empty:
        print(f"   [skip] не удалось извлечь P/Q")
        continue

    sess_id = base_t1.strftime("%Y-%m-%d_%H-%M-%S") if base_t1 is not None else p.stem
    mill_records[sess_id] = df
    print(f"   [OK] P/Q: rows={len(df)}, "
          f"t={df.index.min()} ... {df.index.max()}")
    print(f"        P_mean={df['P'].mean():.1f} кВт, "
          f"Q_mean={df['Q'].mean():.1f} квар, "
          f"pf_mean={df['pf'].mean():.3f}")
    print(f"        источники: P из '{df.attrs.get('p_header')}', "
          f"Q из '{df.attrs.get('q_header')}'")

    # ПКЭ — интегральные показатели качества
    pqe = load_mill_pqe(p)
    if pqe is not None and not pqe.empty:
        mill_pqe_records[sess_id] = pqe
        # Сводка ключевых ПКЭ-метрик
        cols_avail = list(pqe.columns)
        summary_keys = [c for c in cols_avail if c.startswith(("K_2U", "K_0U", "U_1", "I_1", "P_1"))][:5]
        if summary_keys:
            stats = ", ".join(f"{c}_mean={pqe[c].mean():.2f}"
                              for c in summary_keys if pqe[c].notna().any())
            print(f"   [OK] ПКЭ: {stats}")

print("\n" + "=" * 78)
print(f"Загружено сессий мощности: {len(mill_records)}")
print(f"Загружено сессий ПКЭ:      {len(mill_pqe_records)}")
print("=" * 78)


### 11.5. Если что-то всё же не загрузилось

Если для какой-то сессии видно `[skip] не удалось извлечь P/Q` или `[WARN] P/Q не опознаны`, посмотрите шапку листа «Мощности»:

```python
sheets = _read_excel_all_sheets(INPUT_DIR / "имя_файла.xlsx")
print(list(sheets.keys()))
print(sheets["Мощности"].iloc[:3, :12])
```

Если шапка нестандартная (например, `P, кВт A`, `P суммарная` вместо обычного `P, Вт`), допишите паттерн в `_P_PATTERNS` либо вызовите ручной загрузчик `load_mill_session_custom()` с явным указанием колонок.

**Файл `160724_170724.xlsx`** — это **сводный файл** с шагом 2 часа (8 строк за 16 часов 16-17 июля). Он покрывает тот же период, что `16.07.2024 16.27.54-17.07.2024 09.09.08.xlsx` (более детально 10-минутный шаг). Скорее всего, его удобно использовать как кросс-проверку, а не как отдельную сессию.

In [ ]:
def inspect_file(name_substring, max_rows=20, max_cols=12, sheet_idx=0):
    """Показывает содержимое первого листа любого файла, имя которого
    содержит указанную подстроку. Используется для ручной разведки структуры.

    Пример:
        inspect_file("160724_170724")
        inspect_file("16.07.2024 06.28")
    """
    cands = [p for p in INPUT_DIR.iterdir()
             if p.is_file() and name_substring.lower() in p.name.lower()]
    if not cands:
        print(f"Файлы с '{name_substring}' не найдены в {INPUT_DIR}")
        return
    for p in cands:
        print(f"\n{'='*78}\n{p.name}\n{'='*78}")
        try:
            sheets = _read_excel_all_sheets(p)
        except Exception as exc:
            print(f"  [ERR] {exc}"); continue
        snames = list(sheets.keys())
        print(f"  Листов/таблиц: {len(snames)} -> {snames[:10]}")
        if not snames: continue
        sheet_name = snames[min(sheet_idx, len(snames)-1)]
        df = sheets[sheet_name]
        print(f"\n  ЛИСТ '{sheet_name}', shape={df.shape}")
        print(f"  Первые {max_rows} строк × {max_cols} колонок:\n")
        with pd.option_context("display.max_columns", max_cols,
                               "display.max_rows", max_rows,
                               "display.width", 200,
                               "display.max_colwidth", 25):
            print(df.iloc[:max_rows, :max_cols].to_string())


def load_mill_session_custom(path, sheet_idx=0, header_row=None,
                             dt_col=0, p_col=1, q_col=2, scale_p=1.0, scale_q=1.0):
    """Ручная загрузка одной сессии измерения, когда автоматический парсер
    не справился. Все индексы — целочисленные позиции.

    Пример:
        df = load_mill_session_custom("160724_170724.xls",
                                       sheet_idx=0, header_row=2,
                                       dt_col=0, p_col=5, q_col=6)
    """
    p = Path(path) if Path(path).is_absolute() else (INPUT_DIR / path)
    sheets = _read_excel_all_sheets(p)
    snames = list(sheets.keys())
    df_raw = sheets[snames[sheet_idx]]

    if header_row is None:
        # автоопределение по числу парсимых дат в столбце dt_col
        best_i, best_n = 0, 0
        for i in range(min(15, len(df_raw))):
            col = df_raw.iloc[i:, dt_col]
            n_ok = pd.to_datetime(col.astype(str).str.split(" - ").str[0],
                                  dayfirst=True, errors="coerce").notna().sum()
            if n_ok > best_n:
                best_n, best_i = n_ok, i
        header_row = best_i

    body = df_raw.iloc[header_row:].reset_index(drop=True)
    out = pd.DataFrame(index=_parse_datetime(body.iloc[:, dt_col]))
    out["P"] = _coerce_numeric(body.iloc[:, p_col]).values * scale_p
    out["Q"] = _coerce_numeric(body.iloc[:, q_col]).values * scale_q
    out = out[~out.index.isna()].sort_index()
    if out.empty:
        print(f"[WARN] {p.name}: ни одна строка не распарсилась")
        return None
    out["S"]  = np.sqrt(out["P"]**2 + out["Q"]**2)
    out["pf"] = out["P"] / out["S"].replace(0, np.nan)
    print(f"[OK] {p.name}: rows={len(out)}, "
          f"P_mean={out['P'].mean():.1f}, Q_mean={out['Q'].mean():.1f}")
    return out[["P", "Q", "S", "pf"]]


# Раскомментируйте и используйте после диагностики:
# inspect_file("160724_170724")
# inspect_file("15.07.2024 10.21.31")


## 12. Загрузка данных по расходу руды и расчёт СУЭ

`ruda_*.xlsx` — почасовая (или иной фиксированной частоты) производительность мельницы в т/ч.

Метрика: удельный расход электроэнергии (СУЭ) =  ΣP·Δt / ΣQ_ore , кВт·ч / т.

Загрузчик использует ту же стратегию поиска самого длинного непрерывного блока валидных дат и фильтра диапазона года 1990-2100, что и универсальный загрузчик мощностей. Дополнительно перебираются разные колонки в качестве кандидата на расход руды (т/ч).

In [ ]:
ore_files = discovered[discovered["kind"] == "ore_throughput"]
ore_records = {}


def load_ore_file(path: Path) -> pd.DataFrame | None:
    """Загрузка файла с расходом руды.

    Использует робастную стратегию: перебирает первые 5 столбцов как
    кандидатов на временную метку, фильтрует по диапазону года, ищет
    самый длинный непрерывный блок валидных дат. Расход руды берётся
    из соседнего числового столбца.
    """
    suffix = path.suffix.lower()
    try:
        if suffix in {".csv", ".tsv"}:
            df_raw = _read_csv(path)
            sheet_name = "csv"
        else:
            sheets = _read_excel_all_sheets(path)
            sheet_name, df_raw = next(iter(sheets.items()))
    except Exception as exc:
        print(f"   [ERR] {path.name}: {exc}")
        return None

    if df_raw is None or df_raw.empty or df_raw.shape[1] < 2:
        return None

    n_cols_to_try = min(5, df_raw.shape[1])
    best = None  # (block_start_pos, dt_col, block_length)

    for dt_col in range(n_cols_to_try):
        col = df_raw.iloc[:, dt_col]
        parsed = pd.to_datetime(col.astype(str).str.split(" - ").str[0],
                                dayfirst=True, errors="coerce")
        try:
            yrs = parsed.dt.year
            valid_mask = parsed.notna() & yrs.ge(1990) & yrs.le(2100)
        except Exception:
            continue

        if int(valid_mask.sum()) < 5:
            continue

        best_start, best_len = None, 0
        cur_start, cur_len = None, 0
        for idx, v in enumerate(valid_mask.values):
            if v:
                if cur_start is None:
                    cur_start, cur_len = idx, 1
                else:
                    cur_len += 1
            else:
                if cur_len > best_len:
                    best_start, best_len = cur_start, cur_len
                cur_start, cur_len = None, 0
        if cur_len > best_len:
            best_start, best_len = cur_start, cur_len

        if best_len < 5:
            continue
        if best is None or best_len > best[2]:
            best = (best_start, dt_col, best_len)

    if best is None:
        return None

    data_start, dt_col, _ = best
    body = df_raw.iloc[data_start:].reset_index(drop=True)

    # Перебираем кандидаты на колонку «расход руды» — берём ту, в которой
    # после очистки больше всего валидных чисел в физически разумном диапазоне
    # (расход руды обычно 10-1000 т/ч).
    best_ore_col = None
    best_ore_n = 0
    for cand in range(df_raw.shape[1]):
        if cand == dt_col:
            continue
        vals = _coerce_numeric(body.iloc[:, cand])
        # Разумный диапазон расхода руды: 10..1500 т/ч
        n_ok = int(((vals > 10) & (vals < 1500)).sum())
        if n_ok > best_ore_n:
            best_ore_n = n_ok
            best_ore_col = cand

    if best_ore_col is None or best_ore_n < 5:
        return None

    parsed_idx = _parse_datetime(body.iloc[:, dt_col])
    out = pd.DataFrame(index=parsed_idx)
    out["ore_tph"] = _coerce_numeric(body.iloc[:, best_ore_col]).values
    out = out[~out.index.isna()].sort_index()
    out = out.dropna(subset=["ore_tph"])

    out.attrs["sheet"] = sheet_name
    out.attrs["ore_col"] = best_ore_col
    out.attrs["dt_col"] = dt_col
    return out if not out.empty else None


print("=" * 78)
print("ДИАГНОСТИКА ФАЙЛОВ РАСХОДА РУДЫ")
print("=" * 78)

for _, row in ore_files.iterrows():
    p = Path(row["path"])
    print(f"\n--- {p.name} ({p.stat().st_size/1024:.0f} KB) ---")
    fmt = detect_actual_format(p)
    print(f"   реальный формат: {fmt}")

    # Сначала покажем структуру первого листа
    try:
        sheets = _read_excel_all_sheets(p)
        print(f"   найдено листов/таблиц: {len(sheets)} -> {list(sheets.keys())[:5]}")
        first_sheet_name, first_df = next(iter(sheets.items()))
        print(f"   первый лист '{first_sheet_name}': shape={first_df.shape}")
        print(f"   первые 5 строк × 6 колонок:")
        for i, r in first_df.iloc[:5, :6].iterrows():
            vals = [str(v)[:18] for v in list(r.values)[:6]]
            print(f"      row{i}: {vals}")
    except Exception as exc:
        print(f"   [ERR] не удалось прочитать: {exc}")
        continue

    df = load_ore_file(p)
    if df is None or df.empty:
        print(f"   [skip] не удалось выделить колонки (timestamp, ore_tph)")
        print(f"          Попробуйте вручную через inspect_file и определите")
        print(f"          номера колонок (dt_col, ore_col) для load_ore_custom")
    else:
        ore_records[p.stem] = df
        print(f"   [OK]   rows={len(df)}, ore_mean={df['ore_tph'].mean():.1f} т/ч, "
              f"dt_col={df.attrs.get('dt_col')}, ore_col={df.attrs.get('ore_col')}")

print(f"\nЗагружено файлов расхода руды: {len(ore_records)}")


### 12.5. Ручная загрузка расхода руды

Если автоматический загрузчик не справился, используйте `load_ore_custom` с явным указанием колонок (после inspect_file):

In [ ]:
def load_ore_custom(path, sheet_idx=0, header_row=None,
                    dt_col=0, ore_col=1, scale=1.0):
    """Ручная загрузка файла расхода руды с явным указанием колонок."""
    p = Path(path) if Path(path).is_absolute() else (INPUT_DIR / path)
    sheets = _read_excel_all_sheets(p)
    snames = list(sheets.keys())
    df_raw = sheets[snames[sheet_idx]]

    if header_row is None:
        best_i, best_n = 0, 0
        for i in range(min(15, len(df_raw))):
            col = df_raw.iloc[i:, dt_col]
            parsed = pd.to_datetime(col.astype(str).str.split(" - ").str[0],
                                    dayfirst=True, errors="coerce")
            try:
                yrs = parsed.dt.year
                valid_mask = parsed.notna() & yrs.ge(1990) & yrs.le(2100)
            except Exception:
                continue
            n_ok = int(valid_mask.sum())
            if n_ok > best_n:
                best_n, best_i = n_ok, i
        header_row = best_i

    body = df_raw.iloc[header_row:].reset_index(drop=True)
    out = pd.DataFrame(index=_parse_datetime(body.iloc[:, dt_col]))
    out["ore_tph"] = _coerce_numeric(body.iloc[:, ore_col]).values * scale
    out = out[~out.index.isna()].sort_index()
    out = out.dropna(subset=["ore_tph"])
    print(f"[OK] {p.name}: rows={len(out)}, ore_mean={out['ore_tph'].mean():.1f} т/ч")
    return out

# inspect_file("ruda_1")            # посмотреть структуру
# load_ore_custom("ruda_1.xlsx", dt_col=0, ore_col=2)   # пример


In [ ]:
def specific_energy_consumption(power_kw: pd.Series, ore_tph: pd.Series,
                                 freq_hours: float = None) -> pd.Series:
    """СУЭ = P·Δt / Q_ore, кВт·ч/т, на каждый шаг."""
    s_p = power_kw.dropna()
    s_q = ore_tph.dropna()
    common = s_p.index.intersection(s_q.index)
    if common.empty:
        # попытка resample к общему индексу с шагом 1 час
        s_p = s_p.resample("1H").mean()
        s_q = s_q.resample("1H").mean()
        common = s_p.index.intersection(s_q.index)
    if common.empty:
        return pd.Series(dtype=float)
    if freq_hours is None:
        diffs = pd.Series(common).diff().dt.total_seconds() / 3600
        freq_hours = float(diffs.median()) if not diffs.dropna().empty else 1.0
    sec = (s_p.loc[common] * freq_hours) / s_q.loc[common].replace(0, np.nan)
    return sec


# Если есть и P-ряд (например, по ячейке 107 или ввод 1), и расход руды,
# можно посчитать СУЭ. Здесь просто выводим, что доступно.
print("Готово. Применять specific_energy_consumption() к парам "
      "(power, ore) по совпадающему окну дат.")


## 13. Feature engineering

Создаём расширенный набор признаков для прогнозных моделей:

- календарные (год, квартал, месяц, день, день недели, час),
- циклические трансформации sin/cos (час, день недели, месяц),
- лаги P и Q: 1, 6, 12, 24, 48, 72, 168 (часы) или 1, 2, 3, 6, 12 (месяцы),
- скользящие средние и стандартные отклонения,
- агрегаты по предприятию (P_all, Q_all, S_all, pf_all),
- (опционально) лаги соседних вводов / генераций.

In [ ]:
DEFAULT_LAGS_HOURLY  = [1, 6, 12, 24, 48, 72, 96, 120, 144, 168]
DEFAULT_LAGS_MONTHLY = [1, 2, 3, 6, 12]
DEFAULT_ROLL_HOURLY  = [3, 6, 24, 168]
DEFAULT_ROLL_MONTHLY = [3, 6, 12]


def add_calendar_features(df, granularity):
    out = df.copy()
    out["year"]      = out.index.year
    out["quarter"]   = out.index.quarter
    out["month"]     = out.index.month
    out["day"]       = out.index.day
    out["dayofyear"] = out.index.dayofyear
    out["weekday"]   = out.index.weekday
    out["is_weekend"] = (out["weekday"] >= 5).astype(int)
    if granularity == "hourly":
        out["hour"]     = out.index.hour
        out["is_bh"]    = ((out["hour"] >= 8) & (out["hour"] < 20)).astype(int)
        out["hour_sin"] = np.sin(2*np.pi*out["hour"]/24)
        out["hour_cos"] = np.cos(2*np.pi*out["hour"]/24)
    out["month_sin"]   = np.sin(2*np.pi*out["month"]/12)
    out["month_cos"]   = np.cos(2*np.pi*out["month"]/12)
    out["weekday_sin"] = np.sin(2*np.pi*out["weekday"]/7)
    out["weekday_cos"] = np.cos(2*np.pi*out["weekday"]/7)
    return out


def add_lags(df, cols=("P","Q"), lags=None, granularity="hourly"):
    if lags is None:
        lags = DEFAULT_LAGS_HOURLY if granularity == "hourly" else DEFAULT_LAGS_MONTHLY
    out = df.copy()
    for c in cols:
        if c in out.columns:
            for L in lags:
                out[f"{c}_lag_{L}"] = out[c].shift(L)
    return out


def add_rolling(df, cols=("P","Q"), windows=None, granularity="hourly"):
    if windows is None:
        windows = DEFAULT_ROLL_HOURLY if granularity == "hourly" else DEFAULT_ROLL_MONTHLY
    out = df.copy()
    for c in cols:
        if c in out.columns:
            shifted = out[c].shift(1)
            for w in windows:
                out[f"{c}_roll_mean_{w}"] = shifted.rolling(w, min_periods=max(1, w//2)).mean()
                out[f"{c}_roll_std_{w}"]  = shifted.rolling(w, min_periods=max(1, w//2)).std()
    return out


def build_feature_matrix(df, granularity="hourly", target_col="P",
                         panel_p=None, panel_q=None, lags=None,
                         add_other_buses=True, drop_na=True,
                         min_panel_coverage=0.5, verbose=False):
    """Собирает матрицу признаков для ML.

    Важно: панельные источники (panel_p, panel_q) могут иметь разные временные
    окна. Например, у одного источника 5064 точки за полугодие, у другого — всего
    52 (за 2 дня в том же окне). Без выравнивания и отсева панельные лаги будут
    NaN почти везде, и dropna() в конце убьёт все строки.

    Стратегия:
      1. reindex(df.index) приводит каждую панельную серию к временному индексу
         текущего df.
      2. Колонки с покрытием < min_panel_coverage отсеиваются как 'сироты'.
      3. Лаги добавляются только для выживших колонок.
    """
    fm = add_calendar_features(df, granularity)
    fm = add_lags(fm, ("P","Q"), lags, granularity)
    fm = add_rolling(fm, ("P","Q"), None, granularity)

    chosen_lags = lags or (DEFAULT_LAGS_HOURLY if granularity == "hourly"
                                                else DEFAULT_LAGS_MONTHLY)

    n_panel_cols_kept = 0
    n_panel_cols_dropped = 0

    if add_other_buses and panel_p is not None:
        own = df.attrs.get("source")
        # Выравниваем панель на временной индекс df
        panel_p_aligned = panel_p.reindex(df.index)
        panel_q_aligned = panel_q.reindex(df.index) if panel_q is not None else None

        # Оставляем только колонки с достаточным покрытием
        good_p = [c for c in panel_p_aligned.columns
                  if c != own and panel_p_aligned[c].notna().mean() >= min_panel_coverage]
        n_panel_cols_kept    += len(good_p)
        n_panel_cols_dropped += len([c for c in panel_p_aligned.columns
                                      if c != own and c not in good_p])

        for col in good_p:
            for L in chosen_lags:
                fm[f"{col}_P_lag_{L}"] = panel_p_aligned[col].shift(L)

        if panel_q_aligned is not None:
            good_q = [c for c in panel_q_aligned.columns
                      if c != own and panel_q_aligned[c].notna().mean() >= min_panel_coverage]
            n_panel_cols_kept    += len(good_q)
            n_panel_cols_dropped += len([c for c in panel_q_aligned.columns
                                          if c != own and c not in good_q])

            for col in good_q:
                for L in chosen_lags:
                    fm[f"{col}_Q_lag_{L}"] = panel_q_aligned[col].shift(L)

    if verbose:
        print(f"  panel cols kept: {n_panel_cols_kept}, dropped (low coverage): {n_panel_cols_dropped}")

    fm = fm.rename(columns={target_col: "y"})
    return fm.dropna() if drop_na else fm


print("Feature-engineering функции готовы.")
print(f"  Default hourly lags: {DEFAULT_LAGS_HOURLY}")
print(f"  Default rolling windows: {DEFAULT_ROLL_HOURLY}")
print(f"  min_panel_coverage по умолчанию: 50% (отсев 'сирот' с малым покрытием)")


## 14. Бенчмарк 20+ моделей машинного обучения

Сравниваются:

- Naive baselines: Persistence (lag 24), Seasonal Naive (lag 168).
- Линейные: LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge, Huber.
- ML: KNN, Decision Tree, Random Forest, Extra Trees, AdaBoost, Gradient Boosting,
  XGBoost, LightGBM, CatBoost, SVR (RBF), MLP.
- Глубокие сети (опционально, если установлен torch): LSTM, GRU, 1D-CNN, Transformer.
- Классические TS: SARIMAX (если рядов достаточно).

Кросс-валидация: `TimeSeriesSplit` с 5 фолдами.

Метрики: MAE, RMSE, MAPE, sMAPE, R², R²_adj, время обучения, время инференса.

In [ ]:
from dataclasses import dataclass
from typing import Any
import time, subprocess

from sklearn.linear_model    import LinearRegression, Ridge, Lasso, ElasticNet, \
                                     BayesianRidge, HuberRegressor
from sklearn.ensemble        import RandomForestRegressor, ExtraTreesRegressor, \
                                     AdaBoostRegressor, GradientBoostingRegressor
from sklearn.tree            import DecisionTreeRegressor
from sklearn.neighbors       import KNeighborsRegressor
from sklearn.svm             import SVR
from sklearn.neural_network  import MLPRegressor
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics         import (mean_absolute_error, mean_squared_error,
                                     mean_absolute_percentage_error, r2_score)

try:    from xgboost import XGBRegressor; HAS_XGB = True
except: HAS_XGB = False
try:    from lightgbm import LGBMRegressor; HAS_LGB = True
except: HAS_LGB = False
try:    from catboost import CatBoostRegressor; HAS_CAT = True
except: HAS_CAT = False
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    HAS_SARIMAX = True
except: HAS_SARIMAX = False


def detect_gpu():
    """Возвращает имя GPU или None. Сначала проверяем torch (надёжнее),
    потом nvidia-smi."""
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except Exception:
        pass
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=name",
                            "--format=csv,noheader"],
                           capture_output=True, text=True, timeout=5)
        if r.returncode == 0 and r.stdout.strip():
            return r.stdout.strip().splitlines()[0]
    except Exception:
        pass
    return None


GPU_NAME = detect_gpu()
USE_GPU  = GPU_NAME is not None
print(f"XGB={HAS_XGB}  LGB={HAS_LGB}  CAT={HAS_CAT}  SARIMAX={HAS_SARIMAX}")
if USE_GPU:
    print(f"GPU: {GPU_NAME} → XGBoost(device='cuda'), CatBoost(task_type='GPU')")
else:
    print("GPU: не обнаружен → все модели на CPU")


@dataclass
class Bundle:
    name: str
    estimator: Any
    needs_scaling: bool = True
    family: str = "ml"


class _LagBaseline:
    def __init__(self, lag_col): self.lag_col = lag_col
    def fit(self, X, y): return self
    def predict(self, X): return X[self.lag_col].values


def make_models(lag24="P_lag_24", lag168="P_lag_168", use_gpu=USE_GPU):
    M = []
    if lag24:
        M.append(Bundle("Persistence_lag24", _LagBaseline(lag24),  False, "baseline"))
    if lag168:
        M.append(Bundle("SeasonalNaive_lag168", _LagBaseline(lag168), False, "baseline"))
    M += [
        Bundle("LinearReg",   LinearRegression(),                                family="linear"),
        Bundle("Ridge",       Ridge(alpha=1.0, random_state=SEED),               family="linear"),
        Bundle("Lasso",       Lasso(alpha=0.001, max_iter=10000, random_state=SEED), family="linear"),
        Bundle("ElasticNet",  ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=SEED), family="linear"),
        Bundle("BayesianRidge", BayesianRidge(),                                 family="linear"),
        Bundle("Huber",       HuberRegressor(max_iter=500),                      family="linear"),
        Bundle("KNN",         KNeighborsRegressor(n_neighbors=10),               family="ml"),
        Bundle("DecisionTree",DecisionTreeRegressor(max_depth=10, random_state=SEED), False, "ml"),
        Bundle("RandomForest",RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=SEED), False, "ml"),
        Bundle("ExtraTrees",  ExtraTreesRegressor(n_estimators=300, n_jobs=-1, random_state=SEED), False, "ml"),
        Bundle("AdaBoost",    AdaBoostRegressor(n_estimators=200, random_state=SEED), False, "ml"),
        Bundle("GradBoosting",GradientBoostingRegressor(n_estimators=300, max_depth=4,
                                  learning_rate=0.05, random_state=SEED), False, "ml"),
        Bundle("SVR_RBF",     SVR(kernel="rbf", C=10.0, epsilon=0.05),           family="ml"),
        Bundle("MLP",         MLPRegressor(hidden_layer_sizes=(128,64), max_iter=500,
                                  early_stopping=True, random_state=SEED),       family="ml"),
    ]
    if HAS_XGB:
        # XGBoost ≥ 2.0 API: device='cuda' + tree_method='hist' = GPU
        xgb_kwargs = dict(n_estimators=600, max_depth=6, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8,
                          random_state=SEED, verbosity=0,
                          tree_method="hist")
        if use_gpu:
            xgb_kwargs["device"] = "cuda"
        else:
            xgb_kwargs["n_jobs"] = -1
        M.append(Bundle("XGBoost", XGBRegressor(**xgb_kwargs), False, "ml"))
    if HAS_LGB:
        # LightGBM на Colab по умолчанию без GPU-сборки → оставляем CPU (он и так быстрый)
        M.append(Bundle("LightGBM",
                        LGBMRegressor(n_estimators=600, num_leaves=63, learning_rate=0.05,
                                      subsample=0.8, colsample_bytree=0.8,
                                      random_state=SEED, n_jobs=-1, verbose=-1),
                        False, "ml"))
    if HAS_CAT:
        cat_kwargs = dict(iterations=600, depth=6, learning_rate=0.05,
                          random_seed=SEED, verbose=False)
        if use_gpu:
            cat_kwargs["task_type"] = "GPU"
            cat_kwargs["devices"]   = "0"
        M.append(Bundle("CatBoost", CatBoostRegressor(**cat_kwargs), False, "ml"))
    return M


def smape(y_true, y_pred):
    den = (np.abs(y_true) + np.abs(y_pred)) / 2
    m = den != 0
    return float(np.mean(np.abs(y_true[m] - y_pred[m]) / den[m]) * 100)


def adj_r2(r2, n, p):
    return 1 - (1-r2)*(n-1)/(n-p-1) if (n-p-1) > 0 else np.nan


def metrics(y_true, y_pred, n_features):
    return {
        "MAE":      float(mean_absolute_error(y_true, y_pred)),
        "RMSE":     float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE_pct": float(mean_absolute_percentage_error(y_true, y_pred) * 100),
        "sMAPE_pct":smape(y_true, y_pred),
        "R2":       float(r2_score(y_true, y_pred)),
        "R2_adj":   adj_r2(r2_score(y_true, y_pred), len(y_true), n_features),
    }


def cv_evaluate(X, y, bundle, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    folds, fits, preds = [], [], []
    for fold, (tr, te) in enumerate(tscv.split(X)):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]
        try:
            est = (Pipeline([("sc", StandardScaler()), ("m", bundle.estimator)])
                   if bundle.needs_scaling else bundle.estimator)
            t0 = time.time(); est.fit(Xtr, ytr); fits.append(time.time()-t0)
            t1 = time.time(); pred = est.predict(Xte); preds.append(time.time()-t1)
            mt = metrics(yte.values, np.asarray(pred), Xtr.shape[1])
            mt["fold"] = fold
            folds.append(mt)
        except Exception as exc:
            print(f"   [{bundle.name} fold {fold}] {exc}")
    if not folds:
        return None
    fdf = pd.DataFrame(folds)
    return {
        "name": bundle.name, "family": bundle.family,
        "MAE_mean": fdf["MAE"].mean(),  "MAE_std": fdf["MAE"].std(),
        "RMSE_mean":fdf["RMSE"].mean(), "RMSE_std":fdf["RMSE"].std(),
        "MAPE_mean":fdf["MAPE_pct"].mean(),
        "sMAPE_mean":fdf["sMAPE_pct"].mean(),
        "R2_mean":  fdf["R2"].mean(),    "R2_std":  fdf["R2"].std(),
        "R2_adj_mean": fdf["R2_adj"].mean(),
        "fit_time_mean":  float(np.mean(fits)),
        "pred_time_mean": float(np.mean(preds)),
        "fold_table": fdf,
    }


def run_benchmark(X, y, n_splits=5):
    bundles = make_models()
    rows, full = [], {}
    for b in bundles:
        print(f"-> {b.name}")
        r = cv_evaluate(X, y, b, n_splits=n_splits)
        if r:
            full[b.name] = r
            rows.append({k:v for k,v in r.items() if k != "fold_table"})
    leaderboard = pd.DataFrame(rows).sort_values("RMSE_mean").reset_index(drop=True)
    return leaderboard, full, {b.name:b for b in bundles}


print("ML pipeline готов.")


## 15. Запуск ML-бенчмарка по всем почасовым источникам

Для каждого почасового ряда (вводы, ячейки фабрики, генерация) — отдельный прогон.
Выходные таблицы и графики складываются в `OUTPUT_DIR/ml/<entity_period>/`.

In [ ]:
def plot_leaderboard(lb, out_path):
    if lb.empty:
        return
    palette = {"baseline": "#888", "linear": "#1f77b4",
               "ml": "#2ca02c", "deep": "#d62728", "ts": "#9467bd"}
    colors = lb["family"].map(palette).fillna("#777")
    fig, ax = plt.subplots(figsize=(10, max(4, 0.32*len(lb))))
    ax.barh(lb["name"], lb["RMSE_mean"], xerr=lb["RMSE_std"],
            color=colors, ecolor="black", capsize=3)
    ax.set_xlabel("RMSE (mean ± std across folds)")
    ax.invert_yaxis(); ax.grid(axis="x", alpha=0.3)
    ax.set_title("Model comparison: RMSE")
    fig.tight_layout(); fig.savefig(out_path); plt.close(fig)


def plot_fold_box(full, metric, out_path):
    rows = []
    for name, r in full.items():
        for _, x in r["fold_table"].iterrows():
            rows.append({"model": name, metric: x[metric]})
    if not rows: return
    df = pd.DataFrame(rows)
    order = df.groupby("model")[metric].mean().sort_values().index
    fig, ax = plt.subplots(figsize=(10, max(4, 0.4*df["model"].nunique())))
    sns.boxplot(data=df, x=metric, y="model", order=order, orient="h", ax=ax)
    ax.set_title(f"Distribution of {metric} across folds")
    ax.grid(axis="x", alpha=0.3); fig.tight_layout()
    fig.savefig(out_path); plt.close(fig)


def plot_pred_vs_actual(X, y, lb, bundles_dict, out_path, k=3):
    n = len(X); split = int(n*0.8)
    Xtr, Xte = X.iloc[:split], X.iloc[split:]
    ytr, yte = y.iloc[:split], y.iloc[split:]
    top = lb["name"].head(k).tolist()
    fig, axes = plt.subplots(k, 1, figsize=(11, 3*k), sharex=True)
    if k == 1: axes = [axes]
    for ax, name in zip(axes, top):
        b = bundles_dict.get(name)
        if not b: continue
        try:
            est = (Pipeline([("sc", StandardScaler()), ("m", b.estimator)])
                   if b.needs_scaling else b.estimator)
            est.fit(Xtr, ytr); yh = est.predict(Xte)
            ax.plot(yte.index, yte.values, "k", lw=1.0, label="Actual")
            ax.plot(yte.index, yh, color="#d62728", lw=1.0, label=name)
            ax.set_title(f"{name} | RMSE={np.sqrt(mean_squared_error(yte, yh)):.2f}")
            ax.legend(loc="upper right"); ax.grid(alpha=0.3)
        except Exception as exc:
            ax.set_title(f"{name}: {exc}")
    fig.tight_layout(); fig.savefig(out_path); plt.close(fig)


# Соберём панели на час
panel_p_h = assemble_panel(loaded, "hourly", "P")
panel_q_h = assemble_panel(loaded, "hourly", "Q")

print(f"Панель P: {panel_p_h.shape[0]} временных меток × {panel_p_h.shape[1]} источников")
print(f"Покрытие источников в панели P (доля валидных значений):")
for col in panel_p_h.columns:
    cov = panel_p_h[col].notna().mean()
    flag = " ✓" if cov >= 0.5 else " ⚠ (будет отсеян как 'сирота')"
    print(f"   {col:35s}  {cov*100:5.1f}%{flag}")

ml_results = {}
for (ent, per, gran), df in loaded.items():
    if gran != "hourly":
        continue
    label = f"{ent}_{per}"
    ml_dir = OUTPUT_DIR / f"ml/{label}"
    ml_dir.mkdir(parents=True, exist_ok=True)

    # Минимальная длина серии (нужно для лагов 168 + rolling 168 + запас)
    if len(df) < 400:
        print(f"[skip] {label}: слишком короткая серия ({len(df)} строк, нужно ≥ 400)")
        continue

    df.attrs["source"] = label
    fm = build_feature_matrix(df, granularity="hourly", target_col="P",
                              panel_p=panel_p_h, panel_q=panel_q_h,
                              verbose=True)
    if len(fm) < 200:
        print(f"[skip] {label}: после feature-engineering rows={len(fm)} (минимум 200)")
        continue

    print(f"\n=== ML benchmark: {label}  (rows={len(fm)}, features={fm.shape[1]-1}) ===")
    y = fm["y"]
    X = fm.drop(columns=["y"])
    lb, full, bundles_d = run_benchmark(X, y, n_splits=5)

    lb.to_csv(ml_dir / "leaderboard.csv", index=False, float_format="%.5f", encoding="utf-8-sig")
    plot_leaderboard(lb,                           ml_dir / "leaderboard_RMSE.png")
    plot_fold_box(full, "RMSE",                    ml_dir / "fold_box_RMSE.png")
    plot_fold_box(full, "R2",                      ml_dir / "fold_box_R2.png")
    plot_pred_vs_actual(X, y, lb, bundles_d,       ml_dir / "pred_vs_actual_top3.png", k=3)
    ml_results[label] = {"leaderboard": lb, "full": full, "bundles": bundles_d, "X": X, "y": y}
    print(f"  топ-1: {lb.iloc[0]['name']}  RMSE={lb.iloc[0]['RMSE_mean']:.2f}  "
          f"R²={lb.iloc[0]['R2_mean']:.3f}")

print(f"\nML результаты собраны для {len(ml_results)} источников.")


## 16. SHAP-анализ для лучшей модели каждого источника

SHAP помогает интерпретировать вклад признаков (лагов, календаря, кросс-нагрузок) в прогноз.

In [ ]:
try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False
print(f"SHAP доступен: {HAS_SHAP}")


def shap_for_best(label, res, max_sample=1000):
    if not HAS_SHAP: return
    lb = res["leaderboard"]
    if lb.empty: return
    best_name = lb.iloc[0]["name"]
    b = res["bundles"].get(best_name)
    if b is None: return
    X = res["X"]; y = res["y"]
    n = len(X); split = int(n*0.8)
    Xtr, Xte = X.iloc[:split], X.iloc[split:]
    ytr      = y.iloc[:split]
    try:
        if b.needs_scaling:
            sc = StandardScaler().fit(Xtr)
            Xtr_s = pd.DataFrame(sc.transform(Xtr), columns=Xtr.columns, index=Xtr.index)
            Xte_s = pd.DataFrame(sc.transform(Xte), columns=Xte.columns, index=Xte.index)
            b.estimator.fit(Xtr_s, ytr)
            sample = Xte_s.sample(min(max_sample, len(Xte_s)), random_state=SEED)
            explainer = shap.Explainer(b.estimator, sample)
        else:
            b.estimator.fit(Xtr, ytr)
            sample = Xte.sample(min(max_sample, len(Xte)), random_state=SEED)
            explainer = shap.Explainer(b.estimator, sample)
        sv = explainer(sample)
        out_path = OUTPUT_DIR / f"ml/{label}/shap_summary_{best_name}.png"
        plt.figure(figsize=(8, 6))
        shap.summary_plot(sv, sample, show=False)
        plt.tight_layout(); plt.savefig(out_path, dpi=300); plt.close()
        print(f"   SHAP сохранён: {out_path}")
    except Exception as exc:
        print(f"   SHAP error for {label}: {exc}")


for label, res in ml_results.items():
    print(f"SHAP for {label}:")
    shap_for_best(label, res)


## 17. Подбор гиперпараметров через Optuna (LightGBM)

Запускаем 50 trials для лучшей конфигурации LightGBM на каждом почасовом источнике.

In [ ]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except Exception:
    HAS_OPTUNA = False
print(f"Optuna: {HAS_OPTUNA}, LightGBM: {HAS_LGB}")


def optuna_lightgbm(X, y, n_trials=50, n_splits=5):
    if not (HAS_OPTUNA and HAS_LGB): return None
    def objective(trial):
        params = {
            "n_estimators":     trial.suggest_int("n_estimators", 200, 1500, step=100),
            "max_depth":        trial.suggest_int("max_depth", 3, 15),
            "num_leaves":       trial.suggest_int("num_leaves", 15, 255),
            "learning_rate":    trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
            "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "random_state": SEED, "n_jobs": -1, "verbose": -1,
        }
        tscv = TimeSeriesSplit(n_splits=n_splits)
        rmses = []
        for tr, te in tscv.split(X):
            est = LGBMRegressor(**params)
            est.fit(X.iloc[tr], y.iloc[tr])
            pred = est.predict(X.iloc[te])
            rmses.append(np.sqrt(mean_squared_error(y.iloc[te], pred)))
        return float(np.mean(rmses))
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params, study.best_value


for label, res in ml_results.items():
    print(f"Optuna for {label}:")
    out = optuna_lightgbm(res["X"], res["y"], n_trials=50)
    if out is None:
        continue
    best_params, best_rmse = out
    pd.Series(best_params).to_csv(OUTPUT_DIR / f"ml/{label}/optuna_best_lgbm.csv",
                                   header=["value"], encoding="utf-8-sig")
    print(f"   best RMSE = {best_rmse:.3f}")
    print(f"   best params = {best_params}")


## 18. Анализ асимметрии «генерация — нагрузка» (ключевой результат)

Проверяем гипотезу, отмеченную в отчёте: при росте суммарной нагрузки выработка генерации увеличивается с запаздыванием, а при снижении нагрузки — уменьшается быстрее. Используем:

1. **Тест Грэнджера** на причинность между суммарной нагрузкой фабрики (`P_factory_sum`) и выработкой каждой ячейки генерации.
2. **Кросс-корреляцию по знаку приращения** — рассчитываем lag-correlations отдельно по положительным и отрицательным приращениям нагрузки.
3. **Гистограммы реакции** — для каждого приращения нагрузки `ΔP_load` смотрим распределение `ΔP_gen` через k часов.

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

def asymmetry_analysis(panel_p, panel_q, out_dir):
    """Анализирует асимметрию реакции генерации на нагрузку."""
    out_dir.mkdir(parents=True, exist_ok=True)

    # Соберём агрегаты
    factory_cols = [c for c in panel_p.columns if c.startswith(("cell_307", "cell_402"))]
    bus_cols     = [c for c in panel_p.columns if c.startswith(("bus_1",   "bus_2"))]
    gen_cols     = [c for c in panel_p.columns if c.startswith(("gen_cell_2", "gen_cell_23"))]

    if not gen_cols:
        print("Нет данных по генерации — пропускаю анализ асимметрии.")
        return

    # Целевая нагрузка: фабричные ячейки + вводы
    load_cols = factory_cols + bus_cols
    load = panel_p[load_cols].sum(axis=1, min_count=1) if load_cols else None
    gen  = panel_p[gen_cols].sum(axis=1, min_count=1)

    if load is None or load.dropna().empty:
        print("Нет рядов нагрузки — пропускаю анализ.")
        return

    # 1. Granger causality (load -> gen)
    aligned = pd.concat({"gen": gen, "load": load}, axis=1).dropna()
    if len(aligned) < 100:
        print(f"Слишком мало точек для Granger: {len(aligned)}")
        return

    granger_results = []
    for L in [1, 3, 6, 12, 24]:
        try:
            res = grangercausalitytests(aligned[["gen", "load"]], maxlag=L, verbose=False)
            f_stat  = res[L][0]["ssr_ftest"][0]
            p_value = res[L][0]["ssr_ftest"][1]
            granger_results.append({"lag": L, "F": f_stat, "pvalue": p_value})
        except Exception as exc:
            granger_results.append({"lag": L, "err": str(exc)})
    pd.DataFrame(granger_results).to_csv(out_dir / "granger_load_to_gen.csv",
                                          index=False, encoding="utf-8-sig")

    # 2. Asymmetric cross-correlation
    rows = []
    d_load = aligned["load"].diff()
    d_gen  = aligned["gen"].diff()
    pos = d_load > 0
    neg = d_load < 0
    for L in range(0, 25):
        d_gen_lag = d_gen.shift(-L)
        common_pos = pos & d_gen_lag.notna() & d_load.notna()
        common_neg = neg & d_gen_lag.notna() & d_load.notna()
        if common_pos.sum() > 10:
            r_pos = np.corrcoef(d_load[common_pos], d_gen_lag[common_pos])[0, 1]
        else:
            r_pos = np.nan
        if common_neg.sum() > 10:
            r_neg = np.corrcoef(d_load[common_neg], d_gen_lag[common_neg])[0, 1]
        else:
            r_neg = np.nan
        rows.append({"lag": L, "corr_pos_increment": r_pos,
                     "corr_neg_increment": r_neg,
                     "n_pos": int(common_pos.sum()),
                     "n_neg": int(common_neg.sum())})
    asym = pd.DataFrame(rows)
    asym.to_csv(out_dir / "asymmetric_corr.csv", index=False, encoding="utf-8-sig")

    # 3. График asymmetric correlation
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(asym["lag"], asym["corr_pos_increment"], "o-", label="ΔP_load > 0",  color="#d62728")
    ax.plot(asym["lag"], asym["corr_neg_increment"], "s-", label="ΔP_load < 0",  color="#1f77b4")
    ax.axhline(0, color="black", lw=0.5)
    ax.set_xlabel("Lag, hours");  ax.set_ylabel("Cross-correlation Δgen vs Δload")
    ax.set_title("Asymmetric response of cogeneration to load increments")
    ax.grid(alpha=0.3); ax.legend()
    fig.tight_layout()
    fig.savefig(out_dir / "asymmetric_response.png"); plt.close(fig)

    # 4. Двусторонний boxplot реакции
    rows = []
    for L in [1, 3, 6, 12]:
        d_gen_lag = d_gen.shift(-L)
        for sign, mask in (("up", pos), ("down", neg)):
            sub = d_gen_lag[mask].dropna()
            for v in sub.values:
                rows.append({"lag": L, "direction": sign, "delta_gen": v})
    if rows:
        df_box = pd.DataFrame(rows)
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.boxplot(data=df_box, x="lag", y="delta_gen", hue="direction",
                    palette={"up":"#d62728","down":"#1f77b4"}, ax=ax)
        ax.axhline(0, color="black", lw=0.5)
        ax.set_xlabel("Lag, hours"); ax.set_ylabel("Δgen, kW")
        ax.set_title("Distribution of cogeneration response by load-change direction")
        ax.grid(axis="y", alpha=0.3)
        fig.tight_layout()
        fig.savefig(out_dir / "asymmetric_response_box.png"); plt.close(fig)

    print("Анализ асимметрии сохранён.")
    return asym


asym = asymmetry_analysis(panel_p_h, panel_q_h, OUTPUT_DIR / "asymmetry")
if asym is not None:
    print(asym.head(15))


## 19. Количественная оценка резервов энергосбережения

Рассчитываем три резерва, упомянутых в отчётах:

1. **Повышение коэффициента мощности** (текущий 0,71 → целевой 0,95). Снижение полной мощности → снижение потерь в трансформаторах и кабелях. Оценка по упрощённой формуле:
   `ΔP_loss / P_loss = 1 − (cosφ_old / cosφ_new)²`.

2. **Холостое потребление частотного преобразователя** (200 кВт при простое мельницы). При среднем времени простоя `t_idle` часов в году — фиксированная экономия `200 кВт × t_idle`.

3. **Модернизация трансформаторного парка**. Потери холостого хода и КЗ современных сухих трансформаторов на 15–30 % ниже. При суммарной нагрузке трансформаторов `S_total` экономия `~2 % S_total · 8760 ч`.

Для всех расчётов нужен реальный годовой расход активной энергии (`W_year`). Здесь — параметризованный шаблон.

In [ ]:
def reserves_estimation(W_year_kwh: float,
                        cos_phi_current: float = 0.71,
                        cos_phi_target:  float = 0.95,
                        loss_share_grid: float = 0.05,    # 5% сетевых потерь
                        idle_kw: float = 200.0,
                        idle_hours_per_year: float = 1500.0,
                        transformer_share: float = 0.30,  # доля трансформаторов в общей нагрузке
                        transformer_loss_reduction: float = 0.20,  # 20% снижение потерь
                        co2_per_kwh: float = 0.4,         # кгCO2/кВт·ч (РФ средняя)
                        elec_price_rub_kwh: float = 6.0):
    """Возвращает таблицу с оценкой резервов."""
    rows = []

    # 1) Power factor improvement
    delta_loss_share = 1 - (cos_phi_current / cos_phi_target) ** 2
    saving_pf_kwh = W_year_kwh * loss_share_grid * delta_loss_share
    rows.append({
        "reserve": "Power factor 0.71 -> 0.95",
        "saving_kwh_per_year": saving_pf_kwh,
        "saving_pct_of_total": saving_pf_kwh / W_year_kwh * 100,
        "saving_rub_per_year": saving_pf_kwh * elec_price_rub_kwh,
        "co2_avoided_t_per_year": saving_pf_kwh * co2_per_kwh / 1000,
    })

    # 2) Idle VFD
    saving_idle_kwh = idle_kw * idle_hours_per_year
    rows.append({
        "reserve": f"Idle VFD shutdown ({idle_kw} kW x {idle_hours_per_year} h)",
        "saving_kwh_per_year": saving_idle_kwh,
        "saving_pct_of_total": saving_idle_kwh / W_year_kwh * 100,
        "saving_rub_per_year": saving_idle_kwh * elec_price_rub_kwh,
        "co2_avoided_t_per_year": saving_idle_kwh * co2_per_kwh / 1000,
    })

    # 3) Transformer modernization
    saving_tr_kwh = W_year_kwh * transformer_share * loss_share_grid * transformer_loss_reduction
    rows.append({
        "reserve": "Transformer modernization (-20% no-load + load losses)",
        "saving_kwh_per_year": saving_tr_kwh,
        "saving_pct_of_total": saving_tr_kwh / W_year_kwh * 100,
        "saving_rub_per_year": saving_tr_kwh * elec_price_rub_kwh,
        "co2_avoided_t_per_year": saving_tr_kwh * co2_per_kwh / 1000,
    })

    df = pd.DataFrame(rows)
    df.loc["TOTAL"] = df.sum(numeric_only=True)
    df.loc["TOTAL", "reserve"] = "TOTAL"
    return df


# ---- параметры на основе ваших данных ----
# По агрегатам: средняя суммарная активная мощность ~13 МВт ⇒ W_year ≈ 13 МВт × 8760 ч
# Скорректируйте после реального расчёта по panel_p_h.
if not panel_p_h.empty:
    W_year_kwh = float(panel_p_h.sum(axis=1, min_count=1).mean()) * 8760
else:
    W_year_kwh = 13_000 * 8760     # дефолт

print(f"W_year_kwh (оценка)= {W_year_kwh:,.0f} кВт·ч")

reserves = reserves_estimation(W_year_kwh)
reserves.to_csv(OUTPUT_DIR / "tables/energy_savings_reserves.csv",
                float_format="%.2f", encoding="utf-8-sig")
print("\n=== Резервы энергосбережения ===")
display(reserves) if "display" in globals() else print(reserves)


# Waterfall chart
def plot_reserves_waterfall(df, out_path):
    df_data = df[df["reserve"] != "TOTAL"]
    fig, ax = plt.subplots(figsize=(10, 5.5))
    bars = ax.barh(df_data["reserve"], df_data["saving_kwh_per_year"]/1e6,
                   color=["#1f77b4", "#2ca02c", "#d62728"])
    ax.set_xlabel("Energy savings, GWh/year")
    ax.set_title("Quantified energy-savings reserves")
    ax.grid(axis="x", alpha=0.3)
    ax.invert_yaxis()
    for bar, val in zip(bars, df_data["saving_kwh_per_year"]/1e6):
        ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
                f"{val:.2f} GWh", va="center", fontsize=10)
    fig.tight_layout()
    fig.savefig(out_path); plt.close(fig)


plot_reserves_waterfall(reserves, OUTPUT_DIR / "figures/reserves_waterfall.png")
print("Waterfall-график сохранён.")


## 20. Сводный отчёт

Финальная сводная таблица: ключевые числа со всех этапов анализа.

In [ ]:
final_summary = []

# Базовые статистики
for (ent, per, gran), df in loaded.items():
    final_summary.append({
        "metric": f"P_mean ({ent}, {per}, {gran})",
        "value":  df["P"].mean(),
        "unit":   "kW",
    })
    final_summary.append({
        "metric": f"Q_mean ({ent}, {per}, {gran})",
        "value":  df["Q"].mean(),
        "unit":   "kvar",
    })
    final_summary.append({
        "metric": f"pf_mean ({ent}, {per}, {gran})",
        "value":  df["pf"].mean(),
        "unit":   "-",
    })

# ML лучшие модели
for label, res in ml_results.items():
    lb = res["leaderboard"]
    if not lb.empty:
        final_summary.append({"metric": f"Best ML model for {label}",
                              "value": lb.iloc[0]["name"], "unit": "-"})
        final_summary.append({"metric": f"  RMSE",
                              "value": lb.iloc[0]["RMSE_mean"], "unit": "kW"})
        final_summary.append({"metric": f"  R^2",
                              "value": lb.iloc[0]["R2_mean"],   "unit": "-"})

# Резервы
total_row = reserves.loc["TOTAL"]
final_summary.append({"metric": "TOTAL energy savings",
                      "value":  total_row["saving_kwh_per_year"]/1e6, "unit": "GWh/year"})
final_summary.append({"metric": "TOTAL CO2 avoided",
                      "value":  total_row["co2_avoided_t_per_year"], "unit": "tCO2/year"})

final_df = pd.DataFrame(final_summary)
final_df.to_csv(OUTPUT_DIR / "tables/FINAL_SUMMARY.csv", index=False,
                float_format="%.4f", encoding="utf-8-sig")
print("=== ИТОГ ===")
print(final_df.to_string(index=False))
print(f"\nВсе результаты сохранены в: {OUTPUT_DIR}")
print(f"  figures/ — {len(list((OUTPUT_DIR/'figures').glob('*.png')))} PNG файлов")
print(f"  tables/  — {len(list((OUTPUT_DIR/'tables').glob('*.csv')))} CSV файлов")
print(f"  ml/      — папок с моделями: {len(list((OUTPUT_DIR/'ml').iterdir()))}")
